In [171]:
import imageio_ffmpeg
import matplotlib
matplotlib.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

In [172]:
"""
Three-Track Cell Motility Simulation
=====================================

This simulation models how a rod-shaped bacterial cell (like Bacteroidetes)
glides along a surface using a protein called SprB, which acts as a
molecular adhesin — a sticky "foot" that grips the substrate.

HOW GLIDING WORKS (the biological idea):
    - The cell body contains internal helical "conveyor belts" called tracks.
    - SprB proteins sit on these tracks and get carried around the cell surface
      like luggage on an airport carousel.
    - When a SprB molecule passes close to the surface (substrate), it can
      temporarily stick to it (bind), forming a tether between the moving
      belt and the stationary ground.
    - Because the belt keeps moving but the anchor point on the ground is
      fixed, the cell body gets pulled forward — like pushing off a wall.
    - When enough SprBs bind and pull, the cell translates (moves) forward.
    - The geometry of the helical tracks also causes the cell to rotate
      (roll) around its own long axis and occasionally change direction (yaw).

WHAT THIS CODE SIMULATES:
    - A single capsule-shaped cell with three helical tracks.
    - Each track carries multiple SprB proteins around a closed loop.
    - SprBs can bind to the substrate when they are close enough to it,
      and detach either stochastically or when the tether is stretched to
      rupture length.
    - Each bound SprB exerts a spring-like force on the cell — it pulls the
      cell toward the anchor point where it is stuck.
    - The net force and torque from all bound SprBs determines how the cell
      translates and rotates each time step.

FORCE MODEL:
    Each bound SprB is modelled as a tether with constant force magnitude
    F_ADHESION (like a stiff spring at its rupture force). The direction of
    the force points from the SprB's current position on the belt toward
    the anchor point on the ground — i.e., the cell is pulled toward where
    the SprB is stuck.

BINDING MODEL:
    Binding probability scales linearly with proximity to the substrate —
    a SprB directly below the cell axis (height=0) binds with full
    BIND_PROBABILITY, while a SprB at the edge of the zone
    (height=BIND_UNBIND_Z_MAX) has probability ~0. This is a purely
    geometric argument: closer contact = higher binding rate. No molecular
    mechanism is assumed. The effect is to concentrate bound SprBs near
    the bottom of the cell, where the yaw lever arm is near zero —
    favouring straight runs — while SprBs that do bind near the zone edges
    (left/right) still generate turns as before. The mixture of these two
    populations is the geometric origin of heavy-tailed run-length
    distributions.

UNBINDING MODEL:
    Two detachment mechanisms:

    (a) STRETCH RUPTURE — checked immediately after the belt advances each
        SprB in Phase 1, before any force calculation. When the distance
        between the SprB's current belt position and its fixed ground anchor
        reaches L_SPRB (150 nm), the tether ruptures. Placing this check
        inside Phase 1 ensures the force calculation in Phase 3 never sees
        an over-stretched tether, and avoids the step-size overshoot problem
        that occurred when the check ran at the end of the step. The height
        gate (detach when belt position rises above BIND_UNBIND_Z_MAX) has
        been removed — stretch rupture handles all geometry-driven detachment,
        including wrap-around at the poles where the height gate failed to
        fire correctly.

    (b) STOCHASTIC — fixed UNBIND_PROBABILITY per timestep, checked in
        Phase 2. No height guard: the physical interpretation is thermal
        detachment, which is independent of belt position height.
        Height-dependent unbinding was tested and reverted: it made
        bottom-bound SprBs too long-lived and too rare, reducing tail
        statistics and increasing the power-law exponent.

OUTPUTS:
    - Cell trajectory (x, y over time)
    - Turn events (when and how much the cell changes direction)
    - SprB position tracking over time
    - Summary statistics (speed, turn frequency, turn length)
    - Plots and animations
"""

import numpy as np                              # numerical arrays and math
import matplotlib.pyplot as plt                 # plotting
from mpl_toolkits.mplot3d.art3d import Line3DCollection  # 3D line drawing
from scipy.interpolate import interp1d          # arc-length interpolation along tracks
from matplotlib import animation               # saving animations
from tqdm import tqdm

import os                                       # file system operations (mkdir, path joins)
import csv                                      # writing CSV output files

# ---------------------------------------------------------------------------
# Simulation parameters
# ---------------------------------------------------------------------------

NUM_TRACKS          = 3        # number of helical conveyor-belt tracks on the cell
SPRBS_PER_TRACK     = 7        # how many SprBs ride each track
NUM_CANDIDATE_SPOTS = 10000    # how many candidate starting positions to sample before choosing SPRBS_PER_TRACK
LAM                 = 20.0     # exponential distribution shape for initial SprB placement (higher = more clustered near track start)
TRACK_SPEED         = 2000.0   # nm/s — how fast the belt carries SprBs around the track
TIME_STEP           = 0.06     # s — duration of each simulation step
T_SIM               = 100.0    # s — total simulation time

L_SPRB              = 150.0    # nm — tether stretch at rupture (hard detachment limit, checked in Phase 1)

BIND_UNBIND_Z_MAX   = 150.0    # nm — SprB can only bind while this close to the substrate
BIND_PROBABILITY    = 0.5      # probability per time step that an unbound in-range SprB binds AT height=0
                                # (scales down linearly to 0 at height=BIND_UNBIND_Z_MAX)
UNBIND_PROBABILITY  = 0.01     # probability per time step that a bound SprB stochastically detaches

CELL_X_BEGIN  = 0.0            # nm — position of the head (front) pole of the cell along x
CELL_X_END    = 6000.0         # nm — position of the tail (rear) pole of the cell along x
CELL_RADIUS   = 250.0          # nm — radius of the cylindrical body cross-section
HELIX_TURNS   = 2              # how many full rotations the helical track makes along the cylinder

R = CELL_RADIUS                # shorthand alias so formulas read more cleanly

N_BODY_POINTS       = 600      # number of sample points used to build the helical (cylindrical) part of a track
N_ARC_POINTS        = 300      # number of sample points used to build the hemispherical end-cap arcs
N_TRANSITION_POINTS = 80       # number of sample points for short arc transitions between helix and pole

CYLINDER_X_START  = R                          # nm — x position where the cylinder begins (after head cap)
CYLINDER_X_END    = CELL_X_END - R             # nm — x position where the cylinder ends (before tail cap)
CYLINDER_LENGTH   = CYLINDER_X_END - CYLINDER_X_START   # nm — total length of the cylindrical section

BASE_HELIX_PHASE   = 0   # rad — angular starting phase common to all tracks (offsets them from y-axis)
DEFAULT_POLE_PHASE = 0.0            # rad — default phase at which a track crosses the pole
SUBSTRATE_Z        = 0             # nm — z-coordinate of the flat substrate (ground plane)

TRACK_X_STARTS    = [250, 2083, 3917]    # nm — axial position where each track begins on the cylinder
POLE_R_TARGETS    = [225.0, 140.0, 213.0]  # nm — how close to the pole tip each track approaches (smaller = closer)
TRACK_POLE_PHASES = [0.0, 0.0, np.pi]   # rad — angular phase at which each track arrives at the pole
REVERSE_TRACK_IDXS = {2}               # set of track indices that run in the reverse direction

# ---------------------------------------------------------------------------
# Physical constants
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# Hydrodynamic drag and adhesion force parameters
# ---------------------------------------------------------------------------
#
# This section defines the physical constants used to convert adhesive forces
# from bound SprB molecules into cell rotation and translation.
#
# The simulation assumes LOW REYNOLDS NUMBER motion (overdamped regime),
# which is appropriate for bacteria in water. In this regime:
#
#       torque = drag_coefficient × angular_velocity
#
# so the angular velocity of the cell is determined by the applied torque
# divided by the viscous drag coefficient. Inertia is neglected.
#
# The drag coefficients used here follow the slender-body formulas from:
#
#   Tirado & de la Torre (1980)
#   "Rotational dynamics of rigid, symmetric top macromolecules"
#
# These formulas approximate a bacterium as a rigid rod moving in a viscous fluid.
#


# ---------------------------------------------------------------------------
# Cell geometry (converted to SI units)
# ---------------------------------------------------------------------------
#
# The simulation uses nanometres internally, but drag formulas require metres.
#
# L = cell length
# D = cell diameter
#
L_CELL_M  = CELL_X_END * 1e-9
D_CELL_M  = 2.0 * CELL_RADIUS * 1e-9


# ---------------------------------------------------------------------------
# Adhesion force per SprB
# ---------------------------------------------------------------------------
#
# Each bound SprB is treated as a tether that pulls with approximately constant
# force when stretched near its rupture limit.
#
# Single-molecule adhesion / motor forces in bacteria are typically in the
# range 10–100 pN. We use 10 pN as a conservative estimate.
#
# Units: Newton (N)
#
F_ADHESION = 10e-12


# ---------------------------------------------------------------------------
# Yaw rotational drag (turning left/right)
# ---------------------------------------------------------------------------
#
# Resistance to rotation around the Z axis (changing heading).
#
# Formula for a slender rod rotating end-over-end:
#
#       γ_yaw = π η L³ / (3 (ln(L/D) − 0.5))
#
# Units:
#   N·m·s / rad
#
# Larger cells → much harder to turn (∝ L³).

GAMMA_ROT = 1.1396e-19 

# ---------------------------------------------------------------------------
# Roll rotational drag (spinning around long axis)
# ---------------------------------------------------------------------------
#
# Resistance to rotation around the cell's long axis.
#
# Rolling is much easier than yawing because the lever arm is the radius,
# not the length. We therefore use the same formula but with D instead of L:
#
#       γ_roll ≈ π η D³ / (3 (ln(D/L) − 0.5))
#
# Since D << L, this gives much smaller drag → roll is easier than yaw.
#
# A safety clamp is used to avoid negative or nonphysical values when
# ln(D/L) becomes negative.
#

GAMMA_ROT_ROLL = 4.7785e-21

# print(f"[DRAG] p = L/2R = {_p_ratio:.2f}, p⁻¹ = {_p_inv:.4f}, δ∥ = {_delta_par:.4f}")
print(f"[DRAG] GAMMA_ROT_ROLL = {GAMMA_ROT_ROLL:.4e} N·m·s")
print(f"[DRAG] GAMMA_ROT      = {GAMMA_ROT:.4e} N·m·s")
print(f"[DRAG] Ratio GAMMA_ROT / GAMMA_ROT_ROLL = {GAMMA_ROT / GAMMA_ROT_ROLL:.2f}x")




# Camera angles used for 3D visualisation depth-cueing
CAMERA_ELEV_DEG = 18.0    # degrees elevation above horizontal
CAMERA_AZIM_DEG = -58.0   # degrees azimuth (rotation around vertical axis)

# Transparency settings for depth-cueing (front-facing surfaces are more opaque)
DEPTH_ALPHA_FRONT  = 1.0    # fully opaque for the front-facing side of the cell
DEPTH_ALPHA_BACK   = 0.5    # semi-transparent for the back-facing side
DEPTH_SIGMOID_WIDTH = 0.15  # controls how sharp the front/back transition is


# ---------------------------------------------------------------------------
# Track geometry helpers
# ---------------------------------------------------------------------------

def axial_offset_to_phase(x_start):
    """
    Convert a track's axial starting position (in nm along x) to a
    helix phase offset (in radians).

    Tracks that start further along the cell have advanced further along
    the helix by the time they reach x_start, so their angular phase is
    larger. This keeps the tracks spatially separated around the circumference.
    """
    fraction_along_cylinder = (x_start - CYLINDER_X_START) / CYLINDER_LENGTH   # 0 to 1 along the cylinder
    total_helix_angle = 2.0 * np.pi * HELIX_TURNS                              # total angle the helix sweeps
    return total_helix_angle * fraction_along_cylinder                          # proportional phase offset in radians


TRACK_CONFIGS = []                          # will hold one config dict per track
for _track_index in range(NUM_TRACKS):      # loop over each track index (0, 1, 2)
    TRACK_CONFIGS.append({
        "track_id":      _track_index,                                           # integer ID of this track
        "name":          f"Track_{_track_index + 1}",                            # human-readable name
        "x_start":       TRACK_X_STARTS[_track_index],                          # nm — where this track begins axially
        "phase_offset":  axial_offset_to_phase(TRACK_X_STARTS[_track_index]),   # rad — helix phase offset from x_start
        "inverted":      False,                                                  # not used currently (reserved for flipped tracks)
        "pole_r_target": POLE_R_TARGETS[_track_index],                          # nm — how close to the pole tip this track goes
        "pole_phase":    TRACK_POLE_PHASES[_track_index],                        # rad — angular phase at the pole
        "dir":           -1.0 if (_track_index in REVERSE_TRACK_IDXS) else +1.0  # +1 = forward, -1 = backward belt direction
    })


def arclen_interp(x_coords, y_coords, z_coords):
    """
    Build three interpolating functions (one per coordinate) parameterised
    by arc length rather than by index.

    This lets us query "what is the (x, y, z) position at arc-length s
    along this curve?" — essential for moving SprBs at a fixed speed
    regardless of how densely the curve was sampled.

    Returns: (fx, fy, fz, total_arclength)
        fx(s), fy(s), fz(s) — callable interpolators, s in [0, total_arclength]
        total_arclength      — float, total length of the curve in nm
    """
    x_coords = np.asarray(x_coords, float)   # ensure numpy float arrays
    y_coords = np.asarray(y_coords, float)
    z_coords = np.asarray(z_coords, float)
    segment_lengths = np.sqrt(                # Euclidean distance between consecutive points
        np.diff(x_coords)**2 + np.diff(y_coords)**2 + np.diff(z_coords)**2
    )
    cumulative_arclength = np.concatenate(([0.0], np.cumsum(segment_lengths)))  # [0, d1, d1+d2, ...]
    fx = interp1d(cumulative_arclength, x_coords, kind="linear", fill_value="extrapolate")  # x(s)
    fy = interp1d(cumulative_arclength, y_coords, kind="linear", fill_value="extrapolate")  # y(s)
    fz = interp1d(cumulative_arclength, z_coords, kind="linear", fill_value="extrapolate")  # z(s)
    return (fx, fy, fz, float(cumulative_arclength[-1]))   # return interpolators + total length


def build_segments(track_phase, pole_r_target, pole_phase=DEFAULT_POLE_PHASE):
    """
    Construct the raw (x, y, z) point arrays for each of the 8 geometric
    segments that make up one closed helical track loop on the cell surface.

    A track is a closed loop that:
      1. Spirals forward along the cylinder (F-Body)
      2. Transitions angularly to the tail pole phase (TailTrans)
      3. Arcs around the tail hemisphere (TailArc)
      4. Transitions back to the helix phase on the reverse side (TailTransB)
      5. Spirals backward along the cylinder (B-Body)
      6. Transitions to the head pole phase (HeadTrans)
      7. Arcs around the head hemisphere (HeadArc)
      8. Transitions back to restart the forward helix (HeadTransB)

    Returns a list of 8 tuples: [(x_arr, y_arr, z_arr), ...]
    All coordinates are in nm, in the cell body frame (cell centred at origin).
    """
    # True if the track arrives at the pole at a different angular phase
    # than it has on the cylinder — a short bridging arc is then needed.
    needs_transition_arc = abs(track_phase - pole_phase) > 1e-10

    # --- Segment 0: Forward helix along cylinder ---
    u_body_fwd = np.linspace(0, 1, N_BODY_POINTS)             # parameter from 0 to 1 along cylinder
    x_body_fwd = CYLINDER_X_START + u_body_fwd * CYLINDER_LENGTH  # x increases from head to tail
    psi_body_fwd = track_phase + 2 * np.pi * HELIX_TURNS * u_body_fwd  # helix angle increases with x
    y_body_fwd = R * np.cos(psi_body_fwd)   # y = R*cos(helix_angle), traces the circular cross-section
    z_body_fwd = R * np.sin(psi_body_fwd)   # z = R*sin(helix_angle)

    # Helix angle at the end of the forward cylinder (where it meets the tail cap)
    psi_helix_at_tail = track_phase + 2 * np.pi * HELIX_TURNS
    # Target angle at the tail pole (may differ from helix angle if pole_phase != track_phase)
    psi_pole_at_tail  = pole_phase  + 2 * np.pi * HELIX_TURNS

    # --- Segment 1: Tail transition arc (angular bridge on the end-cap plane) ---
    if needs_transition_arc:
        u = np.linspace(0, 1, N_TRANSITION_POINTS)                      # interpolation parameter
        psi = psi_helix_at_tail + (psi_pole_at_tail - psi_helix_at_tail) * u  # sweep angle from helix to pole phase
        x_tail_trans = np.full_like(u, CYLINDER_X_END)   # stays at the same x (end of cylinder)
        y_tail_trans = R * np.cos(psi)                   # sweeps around circumference at fixed x
        z_tail_trans = R * np.sin(psi)
    else:
        # No transition needed — helix already arrives at the correct pole angle
        x_tail_trans = np.array([CYLINDER_X_END])
        y_tail_trans = np.array([R * np.cos(psi_pole_at_tail)])
        z_tail_trans = np.array([R * np.sin(psi_pole_at_tail)])

    # --- Segment 2: Tail hemisphere arc ---
    u_tail_arc = np.linspace(0, 1, N_ARC_POINTS)                         # 0 = equator, 1 = pole tip
    psi_tail_arc = psi_pole_at_tail + np.pi * u_tail_arc                  # azimuthal angle sweeps pi (half circle around)
    radius_tail_arc = R - (R - pole_r_target) * np.sin(np.pi * u_tail_arc)  # radial distance shrinks toward pole_r_target at the tip
    cos_beta_tail = np.sqrt(1.0 - np.clip(radius_tail_arc / R, 0, 1)**2)  # axial offset from end of cylinder (spherical geometry)
    x_tail_arc = (CELL_X_END - R) + R * cos_beta_tail   # x moves out toward the pole tip
    y_tail_arc = radius_tail_arc * np.cos(psi_tail_arc)  # y component of the arc
    z_tail_arc = radius_tail_arc * np.sin(psi_tail_arc)  # z component of the arc

    # After going over the tail pole, the track is now on the "underside" —
    # angle has shifted by pi, and we need to transition back to the backward helix angle.
    psi_pole_at_tail_end     = psi_pole_at_tail  + np.pi   # angle after crossing the pole
    psi_helix_backward_start = psi_helix_at_tail + np.pi   # where the backward helix begins (opposite side)

    # --- Segment 3: Tail transition arc (back from pole to backward helix) ---
    if needs_transition_arc:
        u = np.linspace(0, 1, N_TRANSITION_POINTS)
        psi = psi_pole_at_tail_end + (psi_helix_backward_start - psi_pole_at_tail_end) * u
        x_tail_trans_b = np.full_like(u, CYLINDER_X_END)
        y_tail_trans_b = R * np.cos(psi)
        z_tail_trans_b = R * np.sin(psi)
    else:
        x_tail_trans_b = np.array([CYLINDER_X_END])
        y_tail_trans_b = np.array([R * np.cos(psi_helix_backward_start)])
        z_tail_trans_b = np.array([R * np.sin(psi_helix_backward_start)])

    # --- Segment 4: Backward helix along cylinder (returning toward head) ---
    u_body_bwd = np.linspace(0, 1, N_BODY_POINTS)
    x_body_bwd = CYLINDER_X_END - u_body_bwd * CYLINDER_LENGTH   # x decreases from tail to head
    psi_body_bwd = psi_helix_backward_start + 2 * np.pi * HELIX_TURNS * u_body_bwd  # helix continues winding
    y_body_bwd = R * np.cos(psi_body_bwd)
    z_body_bwd = R * np.sin(psi_body_bwd)

    # Helix angle at the head end after returning
    psi_helix_at_head = psi_helix_backward_start + 2 * np.pi * HELIX_TURNS
    # Target phase at the head pole (includes full offset from all winding so far)
    psi_pole_at_head  = pole_phase + np.pi + 4 * np.pi * HELIX_TURNS

    # --- Segment 5: Head transition arc ---
    if needs_transition_arc:
        u = np.linspace(0, 1, N_TRANSITION_POINTS)
        psi = psi_helix_at_head + (psi_pole_at_head - psi_helix_at_head) * u
        x_head_trans = np.full_like(u, CYLINDER_X_START)
        y_head_trans = R * np.cos(psi)
        z_head_trans = R * np.sin(psi)
    else:
        x_head_trans = np.array([CYLINDER_X_START])
        y_head_trans = np.array([R * np.cos(psi_pole_at_head)])
        z_head_trans = np.array([R * np.sin(psi_pole_at_head)])

    # --- Segment 6: Head hemisphere arc ---
    u_head_arc = np.linspace(0, 1, N_ARC_POINTS)
    psi_head_arc = psi_pole_at_head + np.pi * u_head_arc
    radius_head_arc = R - (R - pole_r_target) * np.sin(np.pi * u_head_arc)
    cos_beta_head = np.sqrt(1.0 - np.clip(radius_head_arc / R, 0, 1)**2)
    x_head_arc = R - R * cos_beta_head    # x moves inward from x=R toward the head pole tip at x=0
    y_head_arc = radius_head_arc * np.cos(psi_head_arc)
    z_head_arc = radius_head_arc * np.sin(psi_head_arc)

    # After crossing the head pole, prepare to restart the forward helix
    psi_pole_at_head_end      = psi_pole_at_head  + np.pi   # angle after crossing head pole
    psi_helix_forward_restart = psi_helix_at_head + np.pi   # angle to restart forward helix

    # --- Segment 7: Head transition arc (back from head pole to forward helix start) ---
    if needs_transition_arc:
        u = np.linspace(0, 1, N_TRANSITION_POINTS)
        psi = psi_pole_at_head_end + (psi_helix_forward_restart - psi_pole_at_head_end) * u
        x_head_trans_b = np.full_like(u, CYLINDER_X_START)
        y_head_trans_b = R * np.cos(psi)
        z_head_trans_b = R * np.sin(psi)
    else:
        x_head_trans_b = np.array([CYLINDER_X_START])
        y_head_trans_b = np.array([R * np.cos(psi_helix_forward_restart)])
        z_head_trans_b = np.array([R * np.sin(psi_helix_forward_restart)])

    # Return all 8 segments as a list of (x_arr, y_arr, z_arr) tuples
    return [
        (x_body_fwd,     y_body_fwd,     z_body_fwd),      # 0: forward helix
        (x_tail_trans,   y_tail_trans,   z_tail_trans),    # 1: tail transition in
        (x_tail_arc,     y_tail_arc,     z_tail_arc),      # 2: tail hemisphere arc
        (x_tail_trans_b, y_tail_trans_b, z_tail_trans_b),  # 3: tail transition out
        (x_body_bwd,     y_body_bwd,     z_body_bwd),      # 4: backward helix
        (x_head_trans,   y_head_trans,   z_head_trans),    # 5: head transition in
        (x_head_arc,     y_head_arc,     z_head_arc),      # 6: head hemisphere arc
        (x_head_trans_b, y_head_trans_b, z_head_trans_b),  # 7: head transition out
    ]


def build_track(track_phase, pole_r_target, pole_phase=DEFAULT_POLE_PHASE):
    """
    Build the complete closed-loop track as a single concatenated array
    of (x, y, z) points.

    Stitches all 8 segments together end-to-end, dropping the first point
    of each subsequent segment to avoid duplicating the junction point.
    """
    segments = build_segments(track_phase, pole_r_target, pole_phase)
    # First segment keeps all points; subsequent segments drop their first point (it's the last of the previous)
    x_full = np.concatenate([seg[0] if i == 0 else seg[0][1:] for i, seg in enumerate(segments)])
    y_full = np.concatenate([seg[1] if i == 0 else seg[1][1:] for i, seg in enumerate(segments)])
    z_full = np.concatenate([seg[2] if i == 0 else seg[2][1:] for i, seg in enumerate(segments)])
    return x_full, y_full, z_full


def head_pole_arc_yz(pole_r_target, pole_phase=DEFAULT_POLE_PHASE, n_points=300):
    """
    Return just the y, z coordinates of the head hemisphere arc,
    used for 2D pole-view visualisation (looking straight down the x-axis).
    """
    u = np.linspace(0, 1, n_points)     # parameter along the arc
    psi = (pole_phase + np.pi + 4 * np.pi * HELIX_TURNS) + np.pi * u   # angle sweeps around the pole
    radius = R - (R - pole_r_target) * np.sin(np.pi * u)               # radius varies from R at equator to pole_r_target at tip
    return radius * np.cos(psi), radius * np.sin(psi)                   # y and z coordinates


def tail_pole_arc_yz(pole_r_target, pole_phase=DEFAULT_POLE_PHASE, n_points=300):
    """
    Return just the y, z coordinates of the tail hemisphere arc,
    used for 2D pole-view visualisation.
    """
    u = np.linspace(0, 1, n_points)
    psi = (pole_phase + 2 * np.pi * HELIX_TURNS) + np.pi * u
    radius = R - (R - pole_r_target) * np.sin(np.pi * u)
    return radius * np.cos(psi), radius * np.sin(psi)


# Human-readable names for the 8 track segments (index matches build_segments output order)
SEGMENT_LABELS = [
    "F-Body", "TailTrans", "TailArc", "TailTransB",
    "B-Body", "HeadTrans", "HeadArc", "HeadTransB",
]

# Index into the segment list for specific named segments (used to identify where a SprB is)
HEAD_ARC_SEGMENT_INDEX = 6   # segment 6 is the head hemisphere arc
TAIL_ARC_SEGMENT_INDEX = 2   # segment 2 is the tail hemisphere arc
F_BODY_SEGMENT_INDEX   = 0   # segment 0 is the forward helix along the cylinder
HEAD_ARC_LAST_SEG      = 6   # (alias, same as HEAD_ARC_SEGMENT_INDEX)
TAIL_ARC_LAST_SEG      = 2   # (alias, same as TAIL_ARC_SEGMENT_INDEX)


def build_track_model(config, base_phase=BASE_HELIX_PHASE):
    """
    Build a complete, interpolatable track model from a track config dict.

    This converts the raw point arrays from build_segments into arc-length
    interpolators so that any SprB at arc-length position s can quickly get
    its (x, y, z) body-frame coordinates.

    Returns a dict with:
        - metadata (track_id, name, phase, etc.)
        - 'segments': list of (name, fx, fy, fz, length) tuples
        - 'cumL': cumulative arc lengths at segment boundaries
        - 'L_total': total circumference of the closed loop
    """
    total_track_phase = float(base_phase + config["phase_offset"])   # total starting angle in radians
    track_pole_phase  = float(config.get("pole_phase", DEFAULT_POLE_PHASE))  # pole angle for this track

    raw_segments = build_segments(total_track_phase, config["pole_r_target"], track_pole_phase)  # get raw point arrays

    def make_seg_interp(x_pts, y_pts, z_pts):
        """Helper: build arc-length interpolators for one segment, handling degenerate (zero-length) segments."""
        x_pts = np.asarray(x_pts, float)
        y_pts = np.asarray(y_pts, float)
        z_pts = np.asarray(z_pts, float)
        if len(x_pts) < 2:   # degenerate: single point — return constant functions
            return (lambda s: x_pts[0], lambda s: y_pts[0], lambda s: z_pts[0], 0.0)
        return arclen_interp(x_pts, y_pts, z_pts)   # normal case: build proper interpolators

    interpolated_segments = []   # will hold (label, fx, fy, fz, length) for each segment
    for i, (x_pts, y_pts, z_pts) in enumerate(raw_segments):
        fx, fy, fz, seg_len = make_seg_interp(x_pts, y_pts, z_pts)   # build interpolators
        interpolated_segments.append((SEGMENT_LABELS[i], fx, fy, fz, seg_len))   # store with label

    seg_lengths = [s[4] for s in interpolated_segments]   # extract just the lengths
    cumL = np.cumsum([0.0] + seg_lengths)                 # cumulative lengths: [0, L0, L0+L1, ...]

    return {
        "track_id":      config["track_id"],        # integer ID
        "name":          config["name"],            # string name
        "phase":         total_track_phase,         # rad — starting helix angle
        "offset":        config["phase_offset"],    # rad — phase offset from axial position
        "x_start":       config["x_start"],         # nm — axial start of this track
        "inverted":      config["inverted"],        # bool — whether track is flipped (unused)
        "pole_r_target": config["pole_r_target"],   # nm — pole approach radius
        "pole_phase":    track_pole_phase,          # rad — angle at the pole
        "dir":           config["dir"],             # +1 or -1 — belt direction
        "segments":      interpolated_segments,     # list of (name, fx, fy, fz, len) per segment
        "cumL":          cumL,                      # cumulative arc-length boundaries
        "L_total":       float(cumL[-1]),           # total loop circumference in nm
    }


def build_all_track_models(base_phase=BASE_HELIX_PHASE):
    """Build and return a list of track model dicts, one per entry in TRACK_CONFIGS."""
    return [build_track_model(config, base_phase) for config in TRACK_CONFIGS]


TRACK_MODELS     = build_all_track_models()                        # build all three track models at import time
TRACK_BY_ID      = {tm["track_id"]: tm for tm in TRACK_MODELS}    # dict for fast lookup by track_id
F_BODY_ARCLENGTH = TRACK_MODELS[0]["segments"][F_BODY_SEGMENT_INDEX][4]  # arc length of the forward helix segment (used for roll calculation)


def map_arclength_to_position(track_model, arclength_position):
    """
    Given an arc-length position s along a track's closed loop, return the
    body-frame (x, y, z) coordinates and the segment the position falls in.

    Walks through the cumulative segment lengths to find which segment
    contains s, then queries the arc-length interpolators for that segment.

    Returns: (seg_idx, seg_name, x_b, y_b, z_b)
        seg_idx  — int, which of the 8 segments the position is in
        seg_name — str, human-readable segment label
        x_b, y_b, z_b — float, body-frame coordinates in nm
    """
    segments = track_model["segments"]   # list of (name, fx, fy, fz, length)
    cumL     = track_model["cumL"]       # cumulative boundaries: [0, L0, L0+L1, ...]

    for seg_idx in range(len(segments)):   # scan through each segment
        if cumL[seg_idx] <= arclength_position <= cumL[seg_idx + 1]:   # check if s falls in this segment
            seg_name, fx, fy, fz, seg_len = segments[seg_idx]
            if seg_len < 1e-12:   # degenerate segment with no length
                return (seg_idx, seg_name, float(fx(0)), float(fy(0)), float(fz(0)))
            local = min(max(arclength_position - cumL[seg_idx], 0.0), seg_len - 1e-12)  # local arc-length within segment
            return (seg_idx, seg_name, float(fx(local)), float(fy(local)), float(fz(local)))

    # If s is beyond the end (should not happen with modular arithmetic), clamp to last segment end
    last = len(segments) - 1
    seg_name, fx, fy, fz, seg_len = segments[last]
    if seg_len < 1e-12:
        return (last, seg_name, float(fx(0)), float(fy(0)), float(fz(0)))
    return (last, seg_name,
            float(fx(seg_len - 1e-12)),
            float(fy(seg_len - 1e-12)),
            float(fz(seg_len - 1e-12)))


def body_to_world(x_body, y_body, z_body, roll_angle):
    """
    Rotate body-frame (x, y, z) coordinates into world frame by applying
    a roll rotation around the x-axis.

    The cell can spin around its own long axis (roll). In the body frame,
    the tracks are fixed. In the world frame, they rotate with the cell.

    roll_angle: radians — how much the cell has rolled so far

    Returns: (x_world, y_world, z_world) — x is unchanged by roll
    """
    cos_r = np.cos(roll_angle)   # precompute trig for efficiency
    sin_r = np.sin(roll_angle)
    return (
        x_body,                                      # x is the long axis — unaffected by roll
        y_body * cos_r - z_body * sin_r,             # y' = y*cos - z*sin  (rotation matrix row)
        y_body * sin_r + z_body * cos_r,             # z' = y*sin + z*cos  (rotation matrix row)
    )


# ---------------------------------------------------------------------------
# Adhesive stretch force
# ---------------------------------------------------------------------------

def adhesive_tether_force(sprb, cell_x, cell_y, cell_roll_rad):
    """
    Calculate the force vector that a bound SprB exerts on the cell.

    Physical picture:
        When a SprB binds, it anchors to a fixed point on the substrate.
        As the cell moves, the anchor stays put but the SprB's position on
        the belt drifts away — pulling the tether. The tether pulls the
        cell back toward the anchor.

    Here we use a constant-force model (not a spring):
        - Direction: from the current belt position toward the anchor.
        - Magnitude: always F_ADHESION (the stall/rupture force of one tether).

    Args:
        sprb          — dict with current SprB body-frame position and anchor coordinates
        cell_x, cell_y — current cell centroid position in the world frame (nm)
        cell_roll_rad  — current roll angle of the cell (rad)

    Returns: (Fx, Fy, Fz, tether_length_nm)
        Fx, Fy, Fz  — force components on the cell in world frame (N)
        tether_length_nm  — scalar distance between belt position and anchor (nm)
    """
    anchor_x_nm, anchor_y_nm = sprb["ground_anchor"]   # fixed anchor coordinates on substrate (nm)

    # Convert body-frame belt position to world frame using current cell pose
    bx_w, by_w, bz_w = body_to_world(sprb["x_b"], sprb["y_b"], sprb["z_b"], cell_roll_rad)

    # World-frame absolute position of the SprB on the belt
    belt_x_nm = cell_x + bx_w          # cell centroid x + body-frame offset
    belt_y_nm = cell_y + by_w          # cell centroid y + body-frame offset
    belt_z_nm = bz_w + R               # add R because body frame has cell axis at z=0, but substrate is at z = -R below the axis

    # Stretch vector: from anchor (on substrate) to belt position (on cell surface)
    dx_nm = belt_x_nm - anchor_x_nm   # positive = belt is ahead of anchor
    dy_nm = belt_y_nm - anchor_y_nm   # lateral displacement
    dz_nm = belt_z_nm                 # vertical displacement (anchor is at z=0)

    tether_length_nm = np.sqrt(dx_nm**2 + dy_nm**2 + dz_nm**2)   # total distance from anchor to belt

    if tether_length_nm < 1e-12:   # degenerate: belt is exactly at the anchor — no direction, no force
        return 0.0, 0.0, 0.0, 0.0

    # Unit vector pointing from anchor toward belt (direction of stretch)
    ux = dx_nm / tether_length_nm
    uy = dy_nm / tether_length_nm
    uz = dz_nm / tether_length_nm

    # Force on the cell is opposite to the stretch direction (cell pulled toward anchor)
    Fx = -F_ADHESION * ux
    Fy = -F_ADHESION * uy
    Fz = -F_ADHESION * uz

    return Fx, Fy, Fz, float(tether_length_nm)


# ---------------------------------------------------------------------------
# Height-dependent binding probability
# ---------------------------------------------------------------------------

def height_dependent_bind_prob(height_above_substrate_nm, base_prob=BIND_PROBABILITY):
    """
    Binding probability scales linearly with proximity to the substrate.

    A SprB at height=0 (touching the substrate directly below the cell axis)
    binds with the full base_prob. A SprB at height=BIND_UNBIND_Z_MAX (the
    outer edge of the binding zone) has probability 0.

    Justification: this is a purely geometric argument about contact likelihood.
    A protein closer to a flat surface has a higher rate of making productive
    contact with it. No specific molecular mechanism is assumed — the linear
    falloff is the simplest monotonic function consistent with this reasoning.

    Physical consequence: bound SprBs are concentrated near the bottom of the
    cell (z_b ≈ -R, y_b ≈ 0), where the yaw lever arm is near zero. This
    naturally favours straight runs. SprBs that do bind near the zone edges
    (left/right of bottom) have higher lever arms and cause turns. The
    contrast between these two populations — frequent short-lived edge bindings
    causing turns, and rarer but sustained bottom bindings causing straight
    runs — is the geometric source of heavy-tailed run-length distributions.

    Args:
        height_above_substrate_nm — current height of SprB above substrate (nm),
                                    in range [0, BIND_UNBIND_Z_MAX]
        base_prob                 — maximum binding probability (at height=0),
                                    defaults to the global BIND_PROBABILITY

    Returns:
        float in [0, base_prob]
    """
    proximity = 1.0 - (height_above_substrate_nm / BIND_UNBIND_Z_MAX)
    proximity = float(np.clip(proximity, 0.0, 1.0))   # guard against floating point edge cases
    return base_prob * proximity


# ---------------------------------------------------------------------------
# Height-dependent unbinding probability (defined but not used in main loop)
# ---------------------------------------------------------------------------

def height_dependent_unbind_prob(height_above_substrate_nm, base_prob=UNBIND_PROBABILITY):
    """
    Unbinding probability scales linearly with height above the substrate.

    Defined for reference but NOT called in the main simulation loop.
    Testing showed it made bottom-bound SprBs too long-lived and too rare,
    reducing tail statistics and increasing the power-law exponent.
    Stochastic unbinding uses a fixed constant instead.

    Args:
        height_above_substrate_nm — current height of SprB above substrate (nm)
        base_prob                 — maximum unbinding probability (at zone edge)

    Returns:
        float in [0, base_prob]
    """
    remoteness = height_above_substrate_nm / BIND_UNBIND_Z_MAX
    remoteness = float(np.clip(remoteness, 0.0, 1.0))
    return base_prob * remoteness


# ---------------------------------------------------------------------------
# Depth-cueing helpers
# ---------------------------------------------------------------------------

def _camera_direction_yz(elev_deg=CAMERA_ELEV_DEG, azim_deg=CAMERA_AZIM_DEG):
    """
    Compute the y and z components of the camera's viewing direction vector.
    Used to determine which parts of the cell surface face the viewer
    (front) vs. face away (back), for depth-cued transparency.
    """
    elev_r = np.deg2rad(elev_deg)    # elevation angle in radians
    azim_r = np.deg2rad(azim_deg)    # azimuth angle in radians
    # Camera looks from (cy, cz) direction in the yz-plane
    return float(np.cos(elev_r) * np.sin(azim_r)), float(np.sin(elev_r))


def _depth_alpha(y_b, z_b, cy=None, cz=None,
                 alpha_front=DEPTH_ALPHA_FRONT,
                 alpha_back=DEPTH_ALPHA_BACK,
                 sigmoid_width=DEPTH_SIGMOID_WIDTH):
    """
    Compute per-point transparency (alpha) based on whether each point
    faces toward or away from the camera.

    Points whose outward normal (y_b, z_b on a cylinder) points toward
    the camera get alpha = alpha_front (opaque).
    Points facing away get alpha = alpha_back (transparent).
    The transition is smooth via a sigmoid function.

    Args:
        y_b, z_b      — body-frame y and z coordinates of the point(s)
        cy, cz        — camera direction components (computed if not supplied)
        alpha_front   — alpha for front-facing points
        alpha_back    — alpha for back-facing points
        sigmoid_width — controls sharpness of front/back transition

    Returns: alpha value(s), same shape as y_b
    """
    if cy is None or cz is None:
        cy, cz = _camera_direction_yz()   # compute camera direction if not provided
    y_b = np.asarray(y_b, float)
    z_b = np.asarray(z_b, float)
    yz_norm   = np.sqrt(y_b**2 + z_b**2)                    # magnitude of (y, z) = radial distance from axis
    safe_norm = np.where(yz_norm > 1e-9, yz_norm, 1.0)      # avoid division by zero on the axis
    dot       = (y_b * cy + z_b * cz) / safe_norm           # dot product of surface normal with camera direction
    sigmoid   = 1.0 / (1.0 + np.exp(-dot / max(sigmoid_width, 1e-6)))  # smooth step: 0 = back, 1 = front
    alpha     = alpha_back + (alpha_front - alpha_back) * sigmoid        # interpolate between back and front alpha
    return float(alpha) if alpha.ndim == 0 else alpha


# RGB colours for the three tracks (matplotlib tab10 palette)
_TRACK_COLORS_RGB = [
    (0.122, 0.467, 0.706),   # blue  — Track 1
    (1.000, 0.498, 0.055),   # orange — Track 2
    (0.173, 0.627, 0.173),   # green — Track 3
]

_BOUND_RGB   = np.array([0.580, 0.404, 0.741])   # purple — colour for bound SprBs
_UNBOUND_RGB = np.array([1.000, 1.000, 1.000])   # white  — colour for unbound SprBs


def _build_depth_cued_track_collection(config, cy, cz, lw=2.0):
    """
    Build a Line3DCollection for one track with depth-cued transparency.

    Each short line segment of the track gets an alpha value based on
    whether it faces the camera or not — front-facing is opaque,
    back-facing is transparent — giving a 3D depth illusion without
    actual 3D rendering of surfaces.

    Returns a Line3DCollection ready to be added to a 3D axes.
    """
    phase      = float(BASE_HELIX_PHASE + config["phase_offset"])       # total helix phase
    pole_phase = float(config.get("pole_phase", DEFAULT_POLE_PHASE))    # pole phase
    x_arr, y_arr, z_arr = build_track(phase, config["pole_r_target"], pole_phase)  # get track coordinates
    x_um = x_arr / 1000.0   # convert nm → µm for plotting
    y_um = y_arr / 1000.0
    z_um = z_arr / 1000.0
    alphas   = _depth_alpha(y_arr, z_arr, cy=cy, cz=cz)   # per-point alpha values
    base_rgb = _TRACK_COLORS_RGB[config["track_id"] % len(_TRACK_COLORS_RGB)]   # pick track colour
    n = len(x_um)
    # Build list of line segments: each is a pair of consecutive 3D points
    segments_3d = [
        [(x_um[i], y_um[i], z_um[i]), (x_um[i+1], y_um[i+1], z_um[i+1])]
        for i in range(n - 1)
    ]
    seg_alphas = 0.5 * (alphas[:-1] + alphas[1:])   # alpha for each segment = average of its two endpoint alphas
    colors = np.zeros((len(segments_3d), 4), dtype=float)   # RGBA array, one row per segment
    colors[:, 0] = base_rgb[0]   # R channel
    colors[:, 1] = base_rgb[1]   # G channel
    colors[:, 2] = base_rgb[2]   # B channel
    colors[:, 3] = seg_alphas    # A channel (depth-cued)
    return Line3DCollection(segments_3d, colors=colors, linewidths=lw)


# ---------------------------------------------------------------------------
# SprB initialisation
# ---------------------------------------------------------------------------

def sample_exponential_positions(total_arclength, n_candidates, lam, rng):
    """
    Sample n_candidates arc-length positions from a transformed exponential
    distribution on [0, total_arclength].

    The distribution is biased toward s=0 (the start of the track) when
    lam is large — matching the observation that SprBs are denser near the
    head pole in real cells.

    u is uniform on [0,1]; the transformation:
        s = L * (1 - exp(-lam*u)) / (1 - exp(-lam))
    maps u to arc-length values that cluster near s=0.
    """
    u = rng.random(n_candidates)   # draw n_candidates uniform random numbers in [0,1)
    return total_arclength * ((1.0 - np.exp(-lam * u)) / (1.0 - np.exp(-lam)))   # transform to arc-length positions


def initialize_sprbs(rng=None):
    """
    Create the initial list of SprB dicts for all tracks.

    For each track:
      1. Sample NUM_CANDIDATE_SPOTS positions from the exponential distribution.
      2. Randomly choose SPRBS_PER_TRACK of them (without replacement).
      3. Look up body-frame (x, y, z) for each chosen arc-length position.
      4. Create a SprB dict with all relevant fields, initially unbound.

    Returns a flat list of SprB dicts (all tracks combined).
    """
    if rng is None:
        rng = np.random.default_rng()   # create a new random number generator if none provided
    all_sprbs = []   # accumulator for all SprB dicts across all tracks
    for tm in TRACK_MODELS:   # loop over each track model
        track_rng = np.random.default_rng(rng.integers(0, 2**32 - 1))   # per-track sub-RNG for reproducibility
        candidates = sample_exponential_positions(
            tm["L_total"], NUM_CANDIDATE_SPOTS, LAM, track_rng
        )   # sample many candidate positions
        chosen = candidates[
            track_rng.choice(len(candidates), SPRBS_PER_TRACK, replace=False)
        ]   # randomly select SPRBS_PER_TRACK of the candidates
        for s in chosen:   # create a SprB dict for each chosen position
            seg_idx, seg_name, x_b, y_b, z_b = map_arclength_to_position(tm, float(s))   # get body-frame coords
            all_sprbs.append({
                "track_id":      int(tm["track_id"]),   # which track this SprB belongs to
                "track_name":    tm["name"],            # human-readable track name
                "dir":           float(tm["dir"]),      # belt direction (+1 or -1)
                "s":             float(s),              # current arc-length position on the track (nm)
                "seg_idx":       seg_idx,               # which of the 8 segments it's currently on
                "seg_name":      seg_name,              # name of that segment
                "x_b":           x_b,                  # body-frame x coordinate (nm)
                "y_b":           y_b,                  # body-frame y coordinate (nm)
                "z_b":           z_b,                  # body-frame z coordinate (nm)
                "bound":         False,                 # starts unbound (not stuck to substrate)
                "ground_anchor": None,                  # no anchor yet (set when it binds)
            })
    return all_sprbs


# ---------------------------------------------------------------------------
# Geometry helpers
# ---------------------------------------------------------------------------

def compute_ground_anchor(cell_x, cell_y, x_world, y_world, z_world):
    """
    Calculate where a newly bound SprB is anchored on the substrate.

    The simplest physical model: the anchor is directly below the SprB's
    world-frame position (vertical projection onto the substrate z=0 plane).

    Returns (anchor_x_nm, anchor_y_nm) — 2D position on the substrate.
    Note: the anchor's z is implicitly 0 (on the flat substrate).
    """
    anchor_x = cell_x + x_world   # world x = cell centroid x + body-frame offset
    anchor_y = cell_y + y_world   # world y = cell centroid y + body-frame offset
    return (float(anchor_x), float(anchor_y))   # anchor is fixed here from now on


def compute_world_z_substrate(z_world):
    """
    Convert a SprB's world-frame z coordinate to its height above the substrate.

    The cell's long axis sits at height R above the substrate (the cell rests
    on the substrate tangentially). In the body frame, z=0 is the cell axis;
    the substrate-touching point is z = -R. So height above substrate is z + R.
    """
    return z_world + R   # add R to shift from cell-centred to substrate-relative height


def wrap_angle_degrees(angle_deg):
    """
    Wrap an angle (in degrees) to the range [-180, +180].

    Used when computing turn angles: a heading change of 181° is the same
    as -179° (turned slightly left, not all the way right).
    """
    return (angle_deg + 180.0) % 360.0 - 180.0   # shift to [0,360), then back to [-180,180)


def compute_cumulative_distance_xy(x_positions, y_positions):
    """
    Compute cumulative distance travelled in µm along the x-y trajectory.

    At each time step, calculates the straight-line distance moved since
    the previous step and accumulates it.

    Returns an array of the same length as x_positions, starting at 0.0.
    Units are µm (inputs in nm, divided by 1000).
    """
    x_arr = np.asarray(x_positions, float)
    y_arr = np.asarray(y_positions, float)
    steps = np.sqrt(np.diff(x_arr)**2 + np.diff(y_arr)**2)   # distance of each step in nm
    return np.concatenate(([0.0], np.cumsum(steps))) / 1000.0  # cumulative sum, converted to µm


# ---------------------------------------------------------------------------
# Main simulation loop
# ---------------------------------------------------------------------------


def run_single_simulation(seed=None, show_progress=True):
    """
    Run one complete simulation of cell gliding for T_SIM seconds.

    Each time step performs four phases in this order:

      1. TRANSLATE AND ROLL (using torque computed at end of previous step)
         Move the cell first, then immediately run a post-move rupture check
         on all bound SprBs.

         FIX (lag bug): n_currently_bound is re-counted at the START of this
         phase from the actual SprB list. If no SprBs are bound right now,
         the cell does not translate or roll — even if net_tau_roll is nonzero
         from the previous step. This eliminates the one-step lag where the
         cell rolled using torque from SprBs that had already detached.

      2. ADVANCE BELTS + STRETCH RUPTURE
         Move every SprB forward along its track by TRACK_SPEED * dt nm.
         Bisection finds the exact crossing point s_star where stretch == L_SPRB.

      3. BINDING / STOCHASTIC DETACHMENT
         Bind unbound SprBs that are within BIND_UNBIND_Z_MAX of the
         substrate, with height-dependent probability. Apply stochastic
         detachment at fixed UNBIND_PROBABILITY to all currently bound SprBs.

      4. FORCE / TORQUE -> HEADING + ROLL TORQUE UPDATE
         Sum forces and torques from all bound SprBs; update yaw heading.
         Store net_tau_roll for use by Phase 1 next step.

         FIX (cross product): Roll torque per SprB is now computed as:
             tau_roll_i = y_b_m * Fz - z_b_m * Fy
         which is the correct x-axis component of the cross product r x F.
         The old code used Fy * sign(z_b) which had wrong magnitude and
         did not allow cancellation between SprBs on opposite sides of the
         cell. The new formula is summed directly as a torque (N.m), so
         the intermediate net_F_y_roll variable is removed entirely.

    Args:
        seed          -- int or None, random seed for reproducibility
        show_progress -- bool, whether to display a tqdm progress bar

    Returns a dict with trajectory, turn events, speed stats, and debug data.
    """
    rng   = np.random.default_rng(seed)
    sprbs = initialize_sprbs(rng)

    total_steps = int(T_SIM / TIME_STEP)
    dt          = TIME_STEP

    cell_x           = 0.0
    cell_y           = 0.0
    cell_heading_deg = 0.0
    cell_roll_rad    = 0.0

    trajectory_x    = [cell_x]
    trajectory_y    = [cell_y]
    heading_history = [cell_heading_deg]

    turn_events        = []
    turn_count         = 0
    cumulative_dist_nm = 0.0
    last_turn_dist_um  = 0.0

    debug_xb    = []
    debug_yb    = []
    debug_zb    = []
    debug_bound = []

    sprb_tracking = []

    # Diagnostic histories
    force_magnitude_history  = []
    torque_magnitude_history = []
    omega_roll_history       = []
    omega_yaw_history        = []
    n_currently_bound_history = []

    rupture_lengths = []
    rupture_sources = []

    # net_tau_roll: roll torque (N.m) carried from Phase 4 into Phase 1.
    # Replaces the old net_F_y_roll accumulator entirely.
    # Initialised to 0 -- the lag fix ensures it is only USED if SprBs are
    # still bound at the start of the next step.
    net_tau_roll = 0.0

    itr = (tqdm(range(total_steps), desc="Simulation", leave=False)
           if show_progress else range(total_steps))

    for step_number in itr:
        current_time = step_number * dt

        omega_roll_this_step = 0.0   # default: no roll unless SprBs are bound

        # ----------------------------------------------------------------
        # PHASE 1: TRANSLATE AND ROLL THE CELL
        #
        # LAG FIX: recount bound SprBs right now rather than relying on
        # n_bound from the previous step. SprBs may have detached during
        # Phase 3 of the last step, making the stored torque stale.
        # ----------------------------------------------------------------
        n_currently_bound = sum(
            1 for s in sprbs if s["bound"] and s["ground_anchor"] is not None
        )
        n_currently_bound_history.append(n_currently_bound)

        if n_currently_bound > 0:
            hr     = np.deg2rad(cell_heading_deg)
            prev_x = cell_x
            prev_y = cell_y
            cell_x += TRACK_SPEED * dt * np.cos(hr)
            cell_y += TRACK_SPEED * dt * np.sin(hr)
            step_dist           = np.sqrt((cell_x - prev_x)**2 + (cell_y - prev_y)**2)
            cumulative_dist_nm += step_dist

            # --- ACTIVE ROLL ---
            # CROSS PRODUCT FIX: net_tau_roll was computed in Phase 4 as
            #   sum of (y_b_m * Fz - z_b_m * Fy) per bound SprB
            # which is the correct torque around the cell's long (x) axis.
            # Divide by drag to get angular velocity, then integrate over dt.
            omega_roll       = net_tau_roll / GAMMA_ROT_ROLL   # rad/s
            active_roll      = omega_roll * dt                  # rad this step
            cell_roll_rad   += active_roll
            omega_roll_this_step = omega_roll                   # store for diagnostics

            # --------------------------------------------------------
            # POST-MOVE RUPTURE CHECK WITH BISECTION
            # --------------------------------------------------------
            dx     = cell_x - prev_x
            dy     = cell_y - prev_y
            d_roll = active_roll

            for sprb in sprbs:
                if not (sprb["bound"] and sprb["ground_anchor"] is not None):
                    continue

                _, _, _, tether_length_post_move = adhesive_tether_force(
                    sprb, cell_x, cell_y, cell_roll_rad
                )

                if tether_length_post_move < L_SPRB:
                    continue

                prev_roll = cell_roll_rad - d_roll
                _, _, _, tether_length_pre_move = adhesive_tether_force(
                    sprb, prev_x, prev_y, prev_roll
                )

                if tether_length_pre_move >= L_SPRB:
                    sprb["bound"]         = False
                    sprb["ground_anchor"] = None
                    rupture_lengths.append(float(tether_length_pre_move))
                    rupture_sources.append("post-move")
                    continue

                # Bisect over t in [0, 1] -- fraction of the full cell displacement
                t_lo, t_hi = 0.0, 1.0
                for _ in range(20):
                    t_mid  = 0.5 * (t_lo + t_hi)
                    cx_mid = prev_x    + t_mid * dx
                    cy_mid = prev_y    + t_mid * dy
                    cr_mid = prev_roll + t_mid * d_roll
                    _, _, _, tether_length_mid = adhesive_tether_force(
                        sprb, cx_mid, cy_mid, cr_mid
                    )
                    if tether_length_mid < L_SPRB:
                        t_lo = t_mid
                    else:
                        t_hi = t_mid

                cx_star = prev_x    + t_hi * dx
                cy_star = prev_y    + t_hi * dy
                cr_star = prev_roll + t_hi * d_roll
                _, _, _, tether_length_star = adhesive_tether_force(
                    sprb, cx_star, cy_star, cr_star
                )
                sprb["bound"]         = False
                sprb["ground_anchor"] = None
                rupture_lengths.append(float(tether_length_star))
                rupture_sources.append("post-move")

        # ----------------------------------------------------------------
        # PHASE 2: ADVANCE BELTS + STRETCH RUPTURE
        # ----------------------------------------------------------------
        for sprb in sprbs:
            tm    = TRACK_BY_ID[sprb["track_id"]]
            s_old = sprb["s"]
            s_new = (s_old + sprb["dir"] * TRACK_SPEED * dt) % tm["L_total"]

            if not (sprb["bound"] and sprb["ground_anchor"] is not None):
                # Unbound -- just update position, no rupture check needed
                seg_idx, seg_name, x_b, y_b, z_b = map_arclength_to_position(tm, s_new)
                sprb["s"]        = s_new
                sprb["seg_idx"]  = seg_idx
                sprb["seg_name"] = seg_name
                sprb["x_b"]      = x_b
                sprb["y_b"]      = y_b
                sprb["z_b"]      = z_b
                continue

            # Stretch at s_old before advancing
            _, _, _, tether_length_old = adhesive_tether_force(
                sprb, cell_x, cell_y, cell_roll_rad
            )

            # Temporarily move to s_new to measure stretch there
            seg_idx_new, seg_name_new, x_b_new, y_b_new, z_b_new = map_arclength_to_position(tm, s_new)
            sprb["s"]        = s_new
            sprb["seg_idx"]  = seg_idx_new
            sprb["seg_name"] = seg_name_new
            sprb["x_b"]      = x_b_new
            sprb["y_b"]      = y_b_new
            sprb["z_b"]      = z_b_new
            _, _, _, tether_length_new = adhesive_tether_force(
                sprb, cell_x, cell_y, cell_roll_rad
            )

            if tether_length_old >= L_SPRB:
                # Missed rupture -- detach at s_old
                seg_idx_old, seg_name_old, x_b_old, y_b_old, z_b_old = map_arclength_to_position(tm, s_old)
                sprb["s"]        = s_old
                sprb["seg_idx"]  = seg_idx_old
                sprb["seg_name"] = seg_name_old
                sprb["x_b"]      = x_b_old
                sprb["y_b"]      = y_b_old
                sprb["z_b"]      = z_b_old
                sprb["bound"]         = False
                sprb["ground_anchor"] = None
                rupture_lengths.append(float(tether_length_old))
                rupture_sources.append("missed")

            elif tether_length_new >= L_SPRB:
                # Threshold crossed this step -- bisect to find exact point
                ds_full      = sprb["dir"] * TRACK_SPEED * dt
                ds_lo, ds_hi = 0.0, ds_full
                for _ in range(20):
                    ds_mid = 0.5 * (ds_lo + ds_hi)
                    s_mid  = (s_old + ds_mid) % tm["L_total"]
                    si_mid, sn_mid, xb_mid, yb_mid, zb_mid = map_arclength_to_position(tm, s_mid)
                    sprb["s"]        = s_mid
                    sprb["seg_idx"]  = si_mid
                    sprb["seg_name"] = sn_mid
                    sprb["x_b"]      = xb_mid
                    sprb["y_b"]      = yb_mid
                    sprb["z_b"]      = zb_mid
                    _, _, _, tether_length_mid = adhesive_tether_force(sprb, cell_x, cell_y, cell_roll_rad)
                    if tether_length_mid < L_SPRB:
                        ds_lo = ds_mid
                    else:
                        ds_hi = ds_mid

                _, _, _, tether_length_star = adhesive_tether_force(sprb, cell_x, cell_y, cell_roll_rad)
                sprb["bound"]         = False
                sprb["ground_anchor"] = None
                rupture_lengths.append(float(tether_length_star))
                rupture_sources.append("bisection")

            # else: no rupture -- s_new already loaded into sprb, nothing to do

        # ----------------------------------------------------------------
        # PHASE 3: BINDING / STOCHASTIC DETACHMENT
        # ----------------------------------------------------------------
        newly_bound = []
        for sprb in sprbs:
            x_w, y_w, z_w = body_to_world(sprb["x_b"], sprb["y_b"], sprb["z_b"], cell_roll_rad)
            height_above_substrate = compute_world_z_substrate(z_w)

            if not sprb["bound"]:
                if 0.0 <= height_above_substrate <= BIND_UNBIND_Z_MAX:
                    p_bind = height_dependent_bind_prob(height_above_substrate)
                    if rng.random() < p_bind:
                        anchor = compute_ground_anchor(cell_x, cell_y, x_w, y_w, z_w)
                        sprb["bound"]         = True
                        sprb["ground_anchor"] = anchor
                        newly_bound.append(sprb)
            else:
                if rng.random() < UNBIND_PROBABILITY:
                    sprb["bound"]         = False
                    sprb["ground_anchor"] = None

        # ----------------------------------------------------------------
        # PHASE 4: FORCE / TORQUE -> HEADING + ROLL TORQUE UPDATE
        #
        # CROSS PRODUCT FIX: roll torque for each SprB is the x-component
        # of r x F in the body frame:
        #
        #   tau_roll_i = y_b_m * Fz - z_b_m * Fy
        #
        # where y_b_m, z_b_m are body-frame coordinates in metres, and
        # Fy, Fz are world-frame force components (in N).
        #
        # This replaces the old:
        #   net_F_y_roll += Fy * sign(z_b)   <-- wrong approximation
        #
        # net_tau_roll is stored here and used in Phase 1 next step.
        # The lag fix ensures it is only applied if SprBs are still bound.
        # ----------------------------------------------------------------
        net_tau_yaw  = 0.0
        net_tau_roll = 0.0   # accumulate fresh each step from current SprBs
        n_bound      = 0

        net_Fx_total = 0.0
        net_Fy_total = 0.0
        net_Fz_total = 0.0

        for sprb in sprbs:
            if sprb["bound"] and sprb["ground_anchor"] is not None:
                n_bound += 1

                Fx, Fy, Fz, _ = adhesive_tether_force(
                    sprb, cell_x, cell_y, cell_roll_rad
                )

                net_Fx_total += Fx
                net_Fy_total += Fy
                net_Fz_total += Fz

                _, y_w, _ = body_to_world(
                    sprb["x_b"], sprb["y_b"], sprb["z_b"], cell_roll_rad
                )

                # Yaw torque: Fx at a y-offset causes turning in the xy plane
                y_contact_m  = y_w * 1e-9   # metres
                net_tau_yaw += Fx * y_contact_m

                # Roll torque: correct cross product, x-component of r x F
                #   r = (x_b, y_b, z_b) in body frame
                #   F = (Fx, Fy, Fz) in world frame
                #   (r x F)_x = y_b * Fz - z_b * Fy
                y_b_m = sprb["y_b"] * 1e-9   # body-frame y in metres
                z_b_m = sprb["z_b"] * 1e-9   # body-frame z in metres
                net_tau_roll += y_b_m * Fz - z_b_m * Fy

        # Diagnostics
        net_force_mag = np.sqrt(net_Fx_total**2 + net_Fy_total**2 + net_Fz_total**2)
        force_magnitude_history.append(float(net_force_mag))
        net_torque_mag = np.sqrt(net_tau_yaw**2 + net_tau_roll**2)
        torque_magnitude_history.append(float(net_torque_mag))

        heading_before = cell_heading_deg
        omega_yaw_step = 0.0
        if n_bound > 0:
            omega_yaw        = net_tau_yaw / GAMMA_ROT
            omega_yaw_step   = float(omega_yaw)
            cell_heading_deg += np.degrees(omega_yaw * dt)

        # print(f"step={step_number} n_currently_bound={n_currently_bound} net_tau_roll={net_tau_roll:.4e} omega_roll={omega_roll_this_step:.2f}")
        omega_roll_history.append(float(omega_roll_this_step))
        omega_yaw_history.append(omega_yaw_step)

        for sprb in newly_bound:
            turn_count += 1
            cum_um = cumulative_dist_nm / 1000.0
            turn_events.append({
                "turn_index":     turn_count,
                "t_s":            float(current_time),
                "x_um":           float(cell_x / 1000.0),
                "y_um":           float(cell_y / 1000.0),
                "cum_dist_um":    float(cum_um),
                "turn_length_um": float(cum_um - last_turn_dist_um),
                "turn_angle_deg": float(wrap_angle_degrees(cell_heading_deg - heading_before)),
                "phi_before_deg": float(heading_before),
                "phi_after_deg":  float(cell_heading_deg),
                "track_id":       int(sprb["track_id"]),
                "track_name":     sprb["track_name"],
                "track_dir":      float(sprb["dir"]),
                "bind_seg":       sprb["seg_name"],
            })
            last_turn_dist_um = cum_um

        trajectory_x.append(cell_x)
        trajectory_y.append(cell_y)
        heading_history.append(cell_heading_deg)
        debug_xb.append(sprbs[0]["x_b"])
        debug_yb.append(sprbs[0]["y_b"])
        debug_zb.append(sprbs[0]["z_b"])
        debug_bound.append(sprbs[0]["bound"])

        for sprb_idx, sprb in enumerate(sprbs):
            xw, yw, zw = body_to_world(sprb["x_b"], sprb["y_b"], sprb["z_b"], cell_roll_rad)
            sprb_tracking.append({
                "step":       step_number,
                "t_s":        float(current_time),
                "sprb_idx":   sprb_idx,
                "track_id":   int(sprb["track_id"]),
                "track_name": sprb["track_name"],
                "x_b":        float(sprb["x_b"]),
                "y_b":        float(sprb["y_b"]),
                "z_b":        float(sprb["z_b"]),
                "seg_name":   sprb["seg_name"],
                "bound":      sprb["bound"],
                "anchor_x":   float(cell_x + xw),
                "anchor_y":   float(cell_y + yw),
                "anchor_z":   float(zw + R),
            })

    # --- Compute summary statistics ---
    time_array    = np.linspace(0, T_SIM, total_steps + 1)
    dist_um       = compute_cumulative_distance_xy(trajectory_x, trajectory_y)
    total_dist_um = float(dist_um[-1])
    avg_speed     = total_dist_um / T_SIM if T_SIM > 0 else 0.0
    total_dist_nm = total_dist_um * 1000.0
    turn_freq     = (turn_count / total_dist_nm * 6000.0) if total_dist_nm > 0 else 0.0

    return {
        "traj_x":                   trajectory_x,
        "traj_y":                   trajectory_y,
        "time":                     time_array,
        "dist_um":                  dist_um,
        "num_turns":                turn_count,
        "turn_freq":                turn_freq,
        "turn_events":              turn_events,
        "heading_hist":             heading_history,
        "avg_speed_um_s":           avg_speed,
        "total_dist_um":            total_dist_um,
        "tracked_xb":               debug_xb,
        "tracked_yb":               debug_yb,
        "tracked_zb":               debug_zb,
        "tracked_bound":            debug_bound,
        "sprbs":                    sprbs,
        "sprb_tracking":            sprb_tracking,
        "rupture_lengths":          rupture_lengths,
        "rupture_sources":          rupture_sources,
        "force_magnitude_history":  force_magnitude_history,
        "torque_magnitude_history": torque_magnitude_history,
        "omega_roll_history":       omega_roll_history,
        "omega_yaw_history":        omega_yaw_history,
        "n_currently_bound_history": n_currently_bound_history,
    }


# ---------------------------------------------------------------------------
# Run-length analysis helpers
# ---------------------------------------------------------------------------

def analyze_run_lengths(result):
    """
    Extract actual bound durations from sprb_tracking data.

    Finds contiguous blocks of bound=True for each SprB and records
    the duration of each block. This measures true tether lifetimes
    rather than the inter-binding-event distances in turn_events.

    Returns a numpy array of bound durations in seconds.
    """
    tracking = result["sprb_tracking"]
    # Group rows by sprb_idx
    from collections import defaultdict
    by_sprb = defaultdict(list)
    for row in tracking:
        by_sprb[row["sprb_idx"]].append((row["t_s"], row["bound"]))

    run_lengths = []
    for sprb_idx, records in by_sprb.items():
        records.sort(key=lambda r: r[0])   # ensure time order
        in_run  = False
        t_start = None
        for t, bound in records:
            if bound and not in_run:
                t_start = t
                in_run  = True
            elif not bound and in_run:
                run_lengths.append(t - t_start)
                in_run = False
        if in_run and t_start is not None:   # SprB still bound at end of simulation
            run_lengths.append(records[-1][0] - t_start)

    return np.array(run_lengths, dtype=float)


def fit_powerlaw_exponent(run_lengths, t_min=None):
    """
    Maximum likelihood estimate of the power-law exponent alpha.

    Uses the Clauset et al. (2009) MLE formula:
        alpha = 1 + n / sum(ln(x_i / x_min))

    This is unbiased and avoids the systematic overestimation that
    comes from fitting a line to a log-log histogram.

    Args:
        run_lengths — array of run durations or distances
        t_min       — lower cutoff for the fit; defaults to the minimum value

    Returns: (alpha_mle, n_samples_above_cutoff)
    """
    r = np.asarray(run_lengths, float)
    r = r[r > 0]   # remove any zero-length runs
    if t_min is None:
        t_min = float(r.min())
    r = r[r >= t_min]
    n = len(r)
    if n < 2:
        return float("nan"), n
    alpha_mle = 1.0 + n / np.sum(np.log(r / t_min))
    return float(alpha_mle), n


# ---------------------------------------------------------------------------
# Plotting / animation helpers
# ---------------------------------------------------------------------------

def _setup_cell_axes(ax):
    """Configure a 3D axes for cell visualisation: labels, aspect ratio, camera, limits."""
    cell_len_um = CELL_X_END / 1000.0    # cell length in µm
    ax.set_xlabel("X (µm)")
    ax.set_ylabel("Y (µm)")
    ax.set_zlabel("Z (µm)")
    ax.set_box_aspect((cell_len_um, 0.6, 0.6))   # stretch x to match actual cell aspect ratio
    ax.view_init(elev=CAMERA_ELEV_DEG, azim=CAMERA_AZIM_DEG)   # set camera angle
    ax.set_xlim(0.0, cell_len_um)
    ax.set_ylim(-CELL_RADIUS / 1000.0, CELL_RADIUS / 1000.0)
    ax.set_zlim(-CELL_RADIUS / 1000.0, CELL_RADIUS / 1000.0)


def _make_cell_figure(figsize=(14, 4)):
    """Create and return a figure + 3D axes with tight layout for cell visualisation."""
    fig = plt.figure(figsize=figsize)
    ax  = fig.add_subplot(111, projection="3d")   # single 3D subplot
    fig.subplots_adjust(left=-0.02, right=1.02, top=1.05, bottom=-0.02)   # maximise plot area
    return fig, ax


def _draw_capsule_surface(ax):
    """
    Draw a semi-transparent capsule (cylinder + two hemispherical end caps)
    on the given 3D axes.

    The capsule represents the cell body. Alpha=0.22 makes it transparent
    enough to see the tracks and SprBs inside.
    """
    theta = np.linspace(0, 2 * np.pi, 100)   # angular parameter for circular cross-section
    x_cyl = np.linspace(R, CELL_X_END - R, 160)   # axial positions along the cylinder
    X, Th = np.meshgrid(x_cyl, theta)   # 2D grids for surface plot
    # Draw the cylindrical body
    ax.plot_surface(X / 1000, R * np.cos(Th) / 1000, R * np.sin(Th) / 1000,
                    alpha=0.22, color='#8a8a8a', linewidth=0)
    pol = np.linspace(0, np.pi / 2, 80)   # polar angle from 0 to pi/2 for a hemisphere
    az  = np.linspace(0, 2 * np.pi, 80)   # azimuthal angle (full circle)
    P, A = np.meshgrid(pol, az)   # 2D grids for hemisphere surface
    # Draw the head (front) hemisphere — centred at x = R (the cylinder start)
    ax.plot_surface((R - R * np.cos(P)) / 1000,
                    (R * np.sin(P) * np.cos(A)) / 1000,
                    (R * np.sin(P) * np.sin(A)) / 1000,
                    alpha=0.22, color='#8a8a8a', linewidth=0)
    # Draw the tail (rear) hemisphere — centred at x = CELL_X_END - R
    ax.plot_surface(((CELL_X_END - R) + R * np.cos(P)) / 1000,
                    (R * np.sin(P) * np.cos(A)) / 1000,
                    (R * np.sin(P) * np.sin(A)) / 1000,
                    alpha=0.22, color='#8a8a8a', linewidth=0)


def _draw_all_tracks(ax):
    """Draw all three tracks on a 3D axes using simple solid lines (no depth cueing)."""
    for config in TRACK_CONFIGS:
        phase      = float(BASE_HELIX_PHASE + config["phase_offset"])   # total helix angle
        pole_phase = float(config.get("pole_phase", DEFAULT_POLE_PHASE))
        tx, ty, tz = build_track(phase, config["pole_r_target"], pole_phase)   # get full track arrays
        ax.plot(tx / 1000, ty / 1000, tz / 1000, lw=2.0)   # plot in µm


def _draw_all_tracks_depth_cued(ax, elev_deg=CAMERA_ELEV_DEG, azim_deg=CAMERA_AZIM_DEG, lw=2.0):
    """Draw all three tracks with depth-cued transparency using Line3DCollections."""
    cy, cz = _camera_direction_yz(elev_deg, azim_deg)   # compute camera direction once
    for config in TRACK_CONFIGS:
        col = _build_depth_cued_track_collection(config, cy, cz, lw=lw)   # build depth-cued collection
        ax.add_collection3d(col)   # add to the 3D axes


def plot_capsule_and_track(sprbs_to_plot, save_path=None, title=None):
    """
    Create a 3D plot of the capsule, all tracks, and a snapshot of SprB positions.

    Bound SprBs are shown in purple; unbound in white.
    Saves to save_path if given, otherwise shows interactively.
    """
    fig = plt.figure(figsize=(18, 6))
    ax  = fig.add_subplot(111, projection='3d')
    _draw_capsule_surface(ax)   # draw the cell body
    _draw_all_tracks(ax)        # draw all three tracks
    for sprb in sprbs_to_plot:
        color = '#9467bd' if sprb["bound"] else 'white'   # purple if bound, white if unbound
        ax.scatter(sprb["x_b"] / 1000, sprb["y_b"] / 1000, sprb["z_b"] / 1000,
                   s=55, edgecolors='k', facecolors=color, linewidths=1.0, zorder=6)
    ax.set_xlabel('X (µm)')
    ax.set_ylabel('Y (µm)')
    ax.set_zlabel('Z (µm)')
    ax.set_box_aspect(((CELL_X_END - CELL_X_BEGIN) / 1000, 0.6, 0.6))
    ax.view_init(elev=CAMERA_ELEV_DEG, azim=CAMERA_AZIM_DEG)
    ax.set_title(title or 'Three-track model: SprBs snapshot')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
    else:
        plt.show()


def plot_pole_views(save_path=None):
    """
    Create a 2D plot showing the track arcs at both poles as seen from the ends of the cell.

    This is a "bull's-eye" view looking straight down the x-axis, showing
    how the three tracks fan out around the circumference at each pole.
    """
    fig, (ax_head, ax_tail) = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
    theta = np.linspace(0, 2 * np.pi, 361)   # full circle for the cell outline
    cy    = R * np.cos(theta) / 1000         # circle y in µm
    cz    = R * np.sin(theta) / 1000         # circle z in µm
    for ax, title in [(ax_head, 'HEAD pole (BODY frame)'), (ax_tail, 'TAIL pole (BODY frame)')]:
        ax.plot(cy, cz, color='#bfbfbf', lw=2)   # draw cell outline circle
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.25)
        ax.set_xlabel('Y (µm)')
        ax.set_ylabel('Z (µm)')
        ax.set_title(title)
        lim = R / 1000 * 1.05    # axis limit slightly beyond the cell radius
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
    for config in TRACK_CONFIGS:
        pp = float(config.get("pole_phase", DEFAULT_POLE_PHASE))
        yh, zh = head_pole_arc_yz(config["pole_r_target"], pp)   # head arc y, z
        yt, zt = tail_pole_arc_yz(config["pole_r_target"], pp)   # tail arc y, z
        ax_head.plot(np.asarray(yh) / 1000, np.asarray(zh) / 1000, lw=2.6)  # plot head arc
        ax_tail.plot(np.asarray(yt) / 1000, np.asarray(zt) / 1000, lw=2.6)  # plot tail arc
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
    else:
        plt.show()


def plot_xy_trajectory(traj_x, traj_y, save_path=None):
    """
    Plot the cell's 2D trajectory (x vs y over the whole simulation).

    Green dot = start, red dot = end.
    """
    plt.figure(figsize=(7.5, 7))
    plt.plot(np.array(traj_x) / 1000, np.array(traj_y) / 1000, lw=2.2, color='#1f77b4')  # trajectory line
    plt.scatter(traj_x[0] / 1000, traj_y[0] / 1000, s=60, c='green', label='start')   # start marker
    plt.scatter(traj_x[-1] / 1000, traj_y[-1] / 1000, s=60, c='red', label='end')     # end marker
    plt.axis('equal')   # equal scaling on x and y so the path isn't distorted
    plt.grid(True, alpha=0.3)
    plt.xlabel('X (µm)')
    plt.ylabel('Y (µm)')
    plt.title('Cell trajectory (x-y)')
    plt.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close()
    else:
        plt.show()


def animate_xy_trajectory(traj_x, traj_y, save_path=None, fps=30, max_frames=200):
    """
    Create an animation of the cell's 2D trajectory, showing the path
    being drawn progressively over time.

    Subsamples the trajectory to at most max_frames frames for efficiency.
    Saves as .mp4 (FFMpeg) or .gif (Pillow) depending on save_path extension.
    """
    x_um = np.asarray(traj_x, float) / 1000   # convert to µm
    y_um = np.asarray(traj_y, float) / 1000
    n_points = len(x_um)
    if n_points < 2:   # nothing to animate
        return
    stride   = max(1, n_points // max_frames)   # subsample stride to keep frame count ≤ max_frames
    x_sub    = x_um[::stride]   # subsampled x coordinates
    y_sub    = y_um[::stride]   # subsampled y coordinates
    n_frames = len(x_sub)       # actual number of frames
    fig, ax = plt.subplots(figsize=(7.5, 7))
    padding = 0.05   # 5% padding around the trajectory bounding box
    x_min, x_max = float(x_um.min()), float(x_um.max())
    y_min, y_max = float(y_um.min()), float(y_um.max())
    xr = x_max - x_min   # x range
    yr = y_max - y_min   # y range
    ax.set_xlim(x_min - padding * xr if xr > 0 else x_min - 1,
                x_max + padding * xr if xr > 0 else x_max + 1)
    ax.set_ylim(y_min - padding * yr if yr > 0 else y_min - 1,
                y_max + padding * yr if yr > 0 else y_max + 1)
    ax.set_aspect('equal')
    ax.grid(False)
    ax.set_xlabel('X (µm)')
    ax.set_ylabel('Y (µm)')
    ax.set_title('Cell trajectory: animated')
    (line,)  = ax.plot([], [], lw=2.2)         # empty line to be filled each frame
    head_dot = ax.scatter([], [], s=40)         # moving dot showing current cell position
    def init():
        """Initialise animation: clear line and dot."""
        line.set_data([], [])
        head_dot.set_offsets(np.c_[[], []])
        return line, head_dot
    def update(frame_idx):
        """Update animation for frame frame_idx: draw path up to this frame."""
        line.set_data(x_sub[:frame_idx + 1], y_sub[:frame_idx + 1])   # draw trajectory so far
        head_dot.set_offsets(np.c_[x_sub[frame_idx], y_sub[frame_idx]])  # move dot to current position
        return line, head_dot
    ani = animation.FuncAnimation(fig, update, frames=n_frames, init_func=init,
                                  interval=1000 / fps, blit=False)
    if save_path is not None:
        if save_path.lower().endswith('.gif'):
            ani.save(save_path, writer=animation.PillowWriter(fps=fps), dpi=90)   # save as GIF
        else:
            ani.save(save_path, writer=animation.FFMpegWriter(fps=fps, bitrate=1800), dpi=150)  # save as video
    else:
        plt.show()
    plt.close(fig)


def animate_sprbs_on_tracks_gif(save_path, seed=0, gif_duration_s=10.0, fps=20, max_frames=250):
    """
    Create an animation of SprBs moving along the tracks on the 3D cell surface.

    This animates only the belt mechanics (SprBs moving on tracks), without
    simulating binding/unbinding or cell motion — it's a visualisation of the
    track geometry and belt speed.

    Bound SprBs are shown in purple; unbound in white. Depth-cueing is applied
    so that SprBs on the far side of the cell appear transparent.
    """
    cy, cz = _camera_direction_yz(CAMERA_ELEV_DEG, CAMERA_AZIM_DEG)   # camera direction for depth-cueing
    rng       = np.random.default_rng(seed)   # seeded RNG for reproducibility
    sprb_list = initialize_sprbs(rng)         # create initial SprBs
    dt        = TIME_STEP
    total_steps   = int(gif_duration_s / dt) + 1    # total steps to simulate
    frame_stride  = max(1, total_steps // max_frames)  # subsample frames
    frame_indices = list(range(0, total_steps, frame_stride))   # which steps to render
    frame_times   = [i * dt for i in frame_indices]   # time label for each frame
    n_frames      = len(frame_times)
    n_sprbs       = len(sprb_list)
    # Pre-allocate arrays for all SprB positions and states at each frame
    X_frames     = np.zeros((n_frames, n_sprbs))       # µm
    Y_frames     = np.zeros((n_frames, n_sprbs))
    Z_frames     = np.zeros((n_frames, n_sprbs))
    Yb_frames    = np.zeros((n_frames, n_sprbs))       # nm (body frame, for depth cueing)
    Zb_frames    = np.zeros((n_frames, n_sprbs))
    Bound_frames = np.zeros((n_frames, n_sprbs), bool) # bound state at each frame
    frame_set = set(frame_indices)   # fast membership test
    fc = 0   # frame counter
    for t in range(total_steps):   # step through time
        for sprb in sprb_list:   # advance each SprB along its track
            tm = TRACK_BY_ID[sprb["track_id"]]
            sprb["s"] = (sprb["s"] + sprb["dir"] * TRACK_SPEED * dt) % tm["L_total"]
            si, sn, x_b, y_b, z_b = map_arclength_to_position(tm, sprb["s"])
            sprb["seg_idx"] = si
            sprb["seg_name"] = sn
            sprb["x_b"] = x_b
            sprb["y_b"] = y_b
            sprb["z_b"] = z_b
        if t in frame_set:   # this step should be saved as a frame
            for j, sprb in enumerate(sprb_list):
                X_frames[fc, j]     = sprb["x_b"] / 1000.0   # convert to µm
                Y_frames[fc, j]     = sprb["y_b"] / 1000.0
                Z_frames[fc, j]     = sprb["z_b"] / 1000.0
                Yb_frames[fc, j]    = sprb["y_b"]   # keep nm for depth-cueing
                Zb_frames[fc, j]    = sprb["z_b"]
                Bound_frames[fc, j] = sprb["bound"]
            fc += 1   # advance frame counter
    # Compute RGBA colours for every SprB at every frame (depth-cued)
    SprB_RGBA = np.zeros((n_frames, n_sprbs, 4), dtype=float)
    for f in range(n_frames):
        alphas = _depth_alpha(Yb_frames[f], Zb_frames[f], cy=cy, cz=cz)   # per-SprB depth alpha
        for j in range(n_sprbs):
            base = _BOUND_RGB if Bound_frames[f, j] else _UNBOUND_RGB   # purple or white base colour
            SprB_RGBA[f, j, :3] = base    # RGB
            SprB_RGBA[f, j, 3]  = alphas[j]  # alpha (depth-cued)
    # Set up the 3D figure
    fig, ax = _make_cell_figure(figsize=(14, 4))
    _draw_capsule_surface(ax)
    _draw_all_tracks_depth_cued(ax, CAMERA_ELEV_DEG, CAMERA_AZIM_DEG, lw=2.0)
    _setup_cell_axes(ax)
    ax.set_xlabel("")   # remove default axis labels (custom text used instead)
    ax.set_ylabel("")
    ax.set_zlabel("")
    # Add custom axis labels as 3D text at specific positions
    ax.text(3.0, -0.55, -0.45, "X (µm)", fontsize=10, ha='center')
    ax.text(7.2,  0.35, -0.35, "Y (µm)", fontsize=10, ha='center')
    ax.text(7.2, -0.45,  0.75, "Z (µm)", fontsize=10, ha='center')
    # Initial scatter plot (frame 0) — will be updated each frame
    scatter = ax.scatter(
        X_frames[0], Y_frames[0], Z_frames[0],
        s=55, c=SprB_RGBA[0], edgecolors="k",
        linewidths=1.0, depthshade=False,   # depthshade=False: use our custom depth-cueing instead
    )
    title_text = ax.set_title(f"SprBs on tracks (t={frame_times[0]:.2f}s)")
    def update_frame(f):
        """Update scatter plot positions and colours for frame f."""
        scatter._offsets3d = (X_frames[f], Y_frames[f], Z_frames[f])  # update 3D positions
        scatter.set_facecolors(SprB_RGBA[f])   # update colours
        scatter.set_edgecolors(np.column_stack([
            np.zeros((n_sprbs, 3)),   # black RGB (0, 0, 0)
            SprB_RGBA[f, :, 3],       # alpha matches the face alpha for consistency
        ]))
        title_text.set_text(f"SprBs on tracks (t={frame_times[f]:.2f}s)")   # update title with time
        return (scatter, title_text)
    ani = animation.FuncAnimation(fig, update_frame, frames=n_frames,
                                  interval=1000 / fps, blit=False)
    _sfkw = {"bbox_inches": "tight", "pad_inches": 0.05}   # tight crop for saving
    if save_path.lower().endswith('.gif'):
        ani.save(save_path, writer=animation.PillowWriter(fps=fps),
                 dpi=90, savefig_kwargs=_sfkw)
    else:
        ani.save(save_path, writer=animation.FFMpegWriter(fps=fps, bitrate=1800),
                 dpi=90, savefig_kwargs=_sfkw)
    plt.close(fig)


def plot_turn_lengths_hist(turn_events_list, save_path=None, bins=30):
    """
    Plot a histogram of the distances between consecutive turn events
    for a single simulation run.

    Each bar represents how many turns were separated by that distance.
    """
    lengths = np.array([e["turn_length_um"] for e in turn_events_list], float)  # extract turn lengths
    plt.figure(figsize=(7, 4))
    if lengths.size:   # only plot if there are any turns
        plt.hist(lengths, bins=bins)
    plt.title("Turn length histogram (this run)" if lengths.size
              else "Turn length histogram (no turns)")
    plt.xlabel("Turn length (µm)")
    plt.ylabel("Count")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close()
    else:
        plt.show()


def plot_all_turn_lengths_hist(all_lengths, save_path=None, bins=50):
    """
    Plot a log-log histogram of turn lengths pooled across ALL simulation runs.

    Log-log axes reveal whether the distribution follows a power law,
    which would appear as a straight line on a log-log plot.
    """
    all_lengths = np.asarray(all_lengths, float)
    plt.figure(figsize=(7.5, 4.5))
    if all_lengths.size:
        plt.hist(all_lengths, bins=bins)
    plt.title("Turn length histogram (ALL runs)" if all_lengths.size
              else "All-run turn length histogram (no turns)")
    plt.xscale('log')   # log scale on x (turn lengths span many orders of magnitude)
    plt.yscale('log')   # log scale on y (rare long turns have very small counts)
    plt.xlabel("Turn length (µm)")
    plt.ylabel("Count")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close()
    else:
        plt.show()


# ---------------------------------------------------------------------------
# CSV output helpers
# ---------------------------------------------------------------------------

def write_frame_csv(traj_x, traj_y, output_path=None, particle_id=1,
                    major_axis=6.0, minor_axis=0.5, mass=1.0, rows_accumulator=None):
    """
    Write the trajectory as a CSV in the format used by particle-tracking
    software (e.g. Trackpy), with one row per frame.

    Columns: frame, mass, x (µm), y (µm), Majoraxis, Minoraxis, frame.1, particle.

    If rows_accumulator is a list, rows are also appended to it (for
    batch collection across multiple runs).
    """
    x_um     = np.asarray(traj_x, float) / 1000   # convert nm → µm
    y_um     = np.asarray(traj_y, float) / 1000
    n_frames = min(len(x_um), len(y_um))           # number of rows to write
    fields   = ["frame", "mass", "x", "y", "Majoraxis", "Minoraxis", "frame.1", "particle"]
    fh     = None    # file handle (None if no output_path given)
    writer = None    # CSV writer (None if no output_path given)
    if output_path:
        fh     = open(output_path, "w", newline="")
        writer = csv.DictWriter(fh, fieldnames=fields)
        writer.writeheader()
    try:
        for i in range(n_frames):   # one row per time step
            row = {
                "frame":     i,                   # frame index (integer)
                "mass":      float(mass),          # placeholder mass value
                "x":         float(x_um[i]),       # x position in µm
                "y":         float(y_um[i]),       # y position in µm
                "Majoraxis": float(major_axis),    # cell major axis length in µm (fixed)
                "Minoraxis": float(minor_axis),    # cell minor axis length in µm (fixed)
                "frame.1":   i,                   # duplicate frame column (required by some software)
                "particle":  int(particle_id),    # which particle/run this row belongs to
            }
            if writer:
                writer.writerow(row)   # write to file
            if rows_accumulator is not None:
                rows_accumulator.append(row)   # also add to the batch accumulator list
    finally:
        if fh:
            fh.close()   # always close the file, even if an exception occurred


def write_sprb_tracking_csv(sprb_tracking, output_path):
    """
    Write the full per-step per-SprB state log to a CSV file.

    Each row records one SprB's body-frame position, segment, and bound state
    at one time step. This file can be large for long simulations with many SprBs.
    """
    fields = ["step", "t_s", "sprb_idx", "track_id", "track_name",
              "x_b", "y_b", "z_b", "seg_name", "bound",
              "anchor_x", "anchor_y", "anchor_z"]
    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(sprb_tracking)   # write all rows at once


def sprb_snapshot_at_time(time_stop, seed=0):
    """
    Advance the SprBs forward in time (without binding/cell motion) and
    return their state at time_stop seconds.

    Used to generate visualisation snapshots at specific times (e.g., t=0s
    and t=10s) to show how the belt distributes SprBs over time.
    """
    rng       = np.random.default_rng(seed)   # seeded RNG
    sprb_list = initialize_sprbs(rng)         # initialise SprBs
    for _ in range(int(time_stop / TIME_STEP)):   # advance step by step
        for sprb in sprb_list:
            tm = TRACK_BY_ID[sprb["track_id"]]
            sprb["s"] = (sprb["s"] + sprb["dir"] * TRACK_SPEED * TIME_STEP) % tm["L_total"]
            si, sn, x_b, y_b, z_b = map_arclength_to_position(tm, sprb["s"])
            sprb["seg_idx"] = si
            sprb["seg_name"] = sn
            sprb["x_b"] = x_b
            sprb["y_b"] = y_b
            sprb["z_b"] = z_b
    return sprb_list   # return SprBs at the requested time


# def plot_bound_sprbs_over_time(sprb_tracking, time_array, save_path=None):
#     """
#     Plot the number of bound SprBs over simulation time as a line chart.
#     """
#     from collections import defaultdict

#     bound_counts = defaultdict(int)
#     step_times   = {}
#     for row in sprb_tracking:
#         step = row["step"]
#         step_times[step] = row["t_s"]
#         if row["bound"]:
#             bound_counts[step] += 1

#     if not step_times:
#         print("[INFO] No sprb_tracking data — bound SprBs plot skipped.")
#         return

#     fig, ax = plt.subplots(figsize=(9, 4))

#     steps_sorted = sorted(step_times.keys())
#     t_vals = [step_times[s] for s in steps_sorted]
#     n_vals = [bound_counts[s] for s in steps_sorted]

#     total_sprbs = len(set(row["sprb_idx"] for row in sprb_tracking))

#     ax.plot(t_vals, n_vals, lw=1.6, color="#1f77b4")
#     ax.set_xlabel("Time (s)")
#     ax.set_ylabel("Number of bound SprBs")
#     ax.set_title(f"Bound SprBs over time (total SprBs = {total_sprbs})")
#     ax.set_ylim(bottom=0)
#     ax.grid(True, alpha=0.3)
#     plt.tight_layout()
#     if save_path:
#         plt.savefig(save_path, dpi=150)
#         plt.close(fig)
#     else:
#         plt.show()

def plot_bound_sprbs_over_time(sprb_tracking, time_array, save_path=None):
    """
    Plot a histogram of bound SprB counts across all time steps.
    """
    from collections import defaultdict

    bound_counts = defaultdict(int)
    step_times   = {}
    for row in sprb_tracking:
        step = row["step"]
        step_times[step] = row["t_s"]
        if row["bound"]:
            bound_counts[step] += 1

    if not step_times:
        print("[INFO] No sprb_tracking data — bound SprBs plot skipped.")
        return

    steps_sorted = sorted(step_times.keys())
    n_vals = [bound_counts[s] for s in steps_sorted]

    total_sprbs = len(set(row["sprb_idx"] for row in sprb_tracking))
    max_bound   = max(n_vals) if n_vals else total_sprbs

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(n_vals, bins=range(0, max_bound + 2), align="left", edgecolor="white")
    ax.set_xlabel("Number of bound SprBs")
    ax.set_ylabel("Number of time steps")
    ax.set_title(f"Distribution of bound SprB count (total SprBs = {total_sprbs})")
    ax.set_xticks(range(0, max_bound + 1))
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
    else:
        plt.show()

def plot_net_force_over_time(force_history, time_array, save_path=None):
    """
    Plot the net adhesive force magnitude over simulation time as a line chart.

    Args:
        force_history — list of net force magnitudes (N) at each time step
        time_array    — 1D array of time values in seconds (length = total_steps + 1)
        save_path     — if given, save the figure to this path; otherwise show interactively
    """
    t_vals = list(time_array[1:len(force_history) + 1])
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(t_vals, force_history, lw=1.6, color="#d62728")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Net force magnitude (N)")
    ax.set_title("Net adhesive force over time")
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
    else:
        plt.show()


def plot_net_torque_over_time(torque_history, time_array, save_path=None):
    """
    Plot the net torque magnitude over simulation time as a line chart.
    Combines yaw torque and roll torque as a vector magnitude.

    Args:
        torque_history — list of net torque magnitudes (N·m) at each time step
        time_array     — 1D array of time values in seconds (length = total_steps + 1)
        save_path      — if given, save the figure to this path; otherwise show interactively
    """
    t_vals = list(time_array[1:len(torque_history) + 1])
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(t_vals, torque_history, lw=1.6, color="#2ca02c")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Net torque magnitude (N·m)")
    ax.set_title("Net torque over time (yaw + roll combined)")
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
    else:
        plt.show()


def plot_omega_roll_over_time(omega_roll_history, time_array, save_path=None):
    """
    Plot the total roll angular velocity (kinematic + active) over simulation time.
    """
    t_vals = list(time_array[1:len(omega_roll_history) + 1])
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(t_vals, omega_roll_history, lw=1.6, color="#9467bd")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Roll angular velocity (rad/s)")
    ax.set_title("Cell roll over time (kinematic + active)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
    else:
        plt.show()


def plot_omega_yaw_over_time(omega_yaw_history, time_array, save_path=None):
    """
    Plot the yaw angular velocity over simulation time.
    """
    t_vals = list(time_array[1:len(omega_yaw_history) + 1])
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(t_vals, omega_yaw_history, lw=1.6, color="#ff7f0e")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Yaw angular velocity (rad/s)")
    ax.set_title("Cell yaw over time")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        plt.close(fig)
    else:
        plt.show()


# ---------------------------------------------------------------------------
# Batch runner
# ---------------------------------------------------------------------------

def run_batch(n_runs, output_base_dir="simulation_output", make_movies=False, plot_track_once=True):
    """
    Run n_runs independent simulations and save all outputs to output_base_dir.

    Each run gets its own subdirectory (run_001/, run_002/, ...) containing:
        - xy_trajectory.png           — 2D trajectory plot
        - trajectory.csv              — per-step position, distance, heading
        - turns.csv                   — per-binding-event details
        - sprb_tracking.csv           — per-step per-SprB state
        - run_summary.csv             — single-row summary statistics
        - turn_lengths_hist.png       — histogram of inter-turn distances
        - bound_sprbs_over_time.png   — line plot of bound SprB count vs time
        - net_force_over_time.png     — line plot of net force magnitude vs time
        - net_force_over_time.csv     — net force magnitude per time step
        - net_torque_over_time.png    — line plot of net torque magnitude vs time
        - net_torque_over_time.csv    — net torque magnitude per time step

    At the top level of output_base_dir, also saves:
        - turn_frequency_summary.csv   — one row per run
        - all_turn_stats_summary.csv   — aggregate statistics across all runs
        - all_turn_lengths_hist.png    — pooled turn-length histogram (log-log)
        - frame_csv.csv               — all runs combined in trackpy format
        - sprbs_on_tracks.mp4         — animation of SprBs on the cell surface
        - powerlaw_exponent.csv       — MLE power-law exponent across all runs

    Args:
        n_runs            — number of independent simulation runs
        output_base_dir   — root directory for all output files
        make_movies       — if True, save per-run trajectory animations (.mp4)
        plot_track_once   — if True, save capsule snapshots and pole views once
    """
    os.makedirs(output_base_dir, exist_ok=True)   # create output directory if it doesn't exist

    # Top-level output file paths
    turn_freq_path = os.path.join(output_base_dir, "turn_frequency_summary.csv")
    all_stats_path = os.path.join(output_base_dir, "all_turn_stats_summary.csv")
    all_hist_path  = os.path.join(output_base_dir, "all_turn_lengths_hist.png")
    frame_csv_path = os.path.join(output_base_dir, "frame_csv.csv")
    powerlaw_path  = os.path.join(output_base_dir, "powerlaw_exponent.csv")

    all_frame_rows       = []   # accumulator for trackpy-format rows across all runs
    all_run_summaries    = []   # one summary dict per run
    all_turn_lengths     = []   # all inter-turn distances pooled across runs (µm)
    all_turn_angles      = []   # all turn angles pooled across runs (degrees)
    all_turn_frequencies = []   # turn frequency per run
    all_average_speeds   = []   # average speed per run (µm/s)
    all_bound_durations  = []   # all bound durations pooled across runs (s)
    batch_rng            = np.random.default_rng()   # master RNG to generate per-run seeds

    if plot_track_once:   # generate one-time visualisation files (not per-run)
        plot_capsule_and_track(sprb_snapshot_at_time(0.0),
                               os.path.join(output_base_dir, "capsule_track_t0s.png"),
                               "Tracks + SprBs at t=0s")
        plot_capsule_and_track(sprb_snapshot_at_time(10.0),
                               os.path.join(output_base_dir, "capsule_track_t10s.png"),
                               "Tracks + SprBs at t=10s")
        plot_pole_views(os.path.join(output_base_dir, "pole_views.png"))

    progress_bar = tqdm(range(1, n_runs + 1), desc="Simulation runs", leave=True)   # outer progress bar

    for run_index in progress_bar:
        seed   = int(batch_rng.integers(0, 2**32 - 1))   # unique random seed for this run
        result = run_single_simulation(seed, show_progress=False)   # run the simulation

        # Unpack results
        traj_x       = result["traj_x"]
        traj_y       = result["traj_y"]
        time_arr     = result["time"]
        dist_um      = result["dist_um"]
        num_turns    = result["num_turns"]
        turn_freq    = result["turn_freq"]
        turn_events  = result["turn_events"]
        heading_hist = result["heading_hist"]
        avg_speed    = result["avg_speed_um_s"]

        all_turn_frequencies.append(float(turn_freq))   # collect for aggregate stats
        all_average_speeds.append(float(avg_speed))

        # Extract bound durations for power-law analysis
        bound_durations = analyze_run_lengths(result)
        all_bound_durations.extend(bound_durations.tolist())

        run_dir = os.path.join(output_base_dir, f"run_{run_index:03d}")   # e.g. simulation_output/run_001/
        os.makedirs(run_dir, exist_ok=True)

        plot_xy_trajectory(traj_x, traj_y, os.path.join(run_dir, "xy_trajectory.png"))   # trajectory plot

        if make_movies:
            animate_xy_trajectory(traj_x, traj_y,
                                  os.path.join(run_dir, "xy_trajectory.mp4"), 20, 250)   # optional animation

        # Write per-step trajectory CSV
        with open(os.path.join(run_dir, "trajectory.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, ["t_s", "x_um", "y_um", "dist_um", "heading_deg"])
            writer.writeheader()
            for i in range(len(time_arr)):
                writer.writerow({
                    "t_s":         float(time_arr[i]),         # time in seconds
                    "x_um":        float(traj_x[i] / 1000),   # x in µm
                    "y_um":        float(traj_y[i] / 1000),   # y in µm
                    "dist_um":     float(dist_um[i]),          # cumulative distance in µm
                    "heading_deg": float(heading_hist[i]),     # heading in degrees
                })

        # --- NEW: write net force CSV ---
        with open(os.path.join(run_dir, "net_force_over_time.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, ["t_s", "net_force_N"])
            writer.writeheader()
            for i, val in enumerate(result["force_magnitude_history"]):
                writer.writerow({"t_s": float(time_arr[i + 1]), "net_force_N": float(val)})

        # --- NEW: write net torque CSV ---
        with open(os.path.join(run_dir, "net_torque_over_time.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, ["t_s", "net_torque_Nm"])
            writer.writeheader()
            for i, val in enumerate(result["torque_magnitude_history"]):
                writer.writerow({"t_s": float(time_arr[i + 1]), "net_torque_Nm": float(val)})

        # Write bound SprBs over time CSV
        with open(os.path.join(run_dir, "bound_sprbs_over_time.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, ["t_s", "n_bound"])
            writer.writeheader()
            for i, val in enumerate(result["n_currently_bound_history"]):
                writer.writerow({"t_s": float(time_arr[i]), "n_bound": int(val)})

        # Write trackpy-format frame CSV for this run (also accumulate for batch file)
        write_frame_csv(traj_x, traj_y, os.path.join(run_dir, "frame_csv.csv"),
                        run_index, rows_accumulator=all_frame_rows)

        # Write per-binding-event turns CSV
        with open(os.path.join(run_dir, "turns.csv"), "w", newline="") as f:
            fields = ["turn_index", "t_s", "x_um", "y_um", "cum_dist_um", "turn_length_um",
                      "turn_angle_deg", "phi_before_deg", "phi_after_deg",
                      "track_id", "track_name", "track_dir", "bind_seg"]
            writer = csv.DictWriter(f, fields)
            writer.writeheader()
            writer.writerows(turn_events)

        # Write full SprB tracking log
        write_sprb_tracking_csv(result["sprb_tracking"],
                                os.path.join(run_dir, "sprb_tracking.csv"))

        # Write single-row summary for this run
        with open(os.path.join(run_dir, "run_summary.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, ["run", "num_turns", "distance_traveled_um",
                                         "turn_frequency", "avg_speed_um_s"])
            writer.writeheader()
            writer.writerow({
                "run":                  run_index,
                "num_turns":            num_turns,
                "distance_traveled_um": float(dist_um[-1]),
                "turn_frequency":       float(turn_freq),
                "avg_speed_um_s":       float(avg_speed),
            })

        plot_turn_lengths_hist(turn_events, os.path.join(run_dir, "turn_lengths_hist.png"))
        plot_bound_sprbs_over_time(result["sprb_tracking"], result["time"], os.path.join(run_dir, "bound_sprbs_over_time.png"))
        # --- NEW: save force and torque plots ---
        plot_net_force_over_time(result["force_magnitude_history"], result["time"], os.path.join(run_dir, "net_force_over_time.png"))
        plot_net_torque_over_time(result["torque_magnitude_history"], result["time"], os.path.join(run_dir, "net_torque_over_time.png"))

        # --- NEW: omega roll and yaw CSVs and plots ---
        with open(os.path.join(run_dir, "omega_roll_over_time.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, ["t_s", "omega_roll_rad_s"])
            writer.writeheader()
            for i, val in enumerate(result["omega_roll_history"]):
                writer.writerow({"t_s": float(time_arr[i]), "omega_roll_rad_s": float(val)})

        with open(os.path.join(run_dir, "omega_yaw_over_time.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, ["t_s", "omega_yaw_rad_s"])
            writer.writeheader()
            for i, val in enumerate(result["omega_yaw_history"]):
                writer.writerow({"t_s": float(time_arr[i + 1]), "omega_yaw_rad_s": float(val)})

        plot_omega_roll_over_time(result["omega_roll_history"], result["time"], os.path.join(run_dir, "omega_roll_over_time.png"))
        plot_omega_yaw_over_time(result["omega_yaw_history"], result["time"], os.path.join(run_dir, "omega_yaw_over_time.png"))

        # Accumulate turn statistics for aggregate analysis
        for e in turn_events:
            all_turn_lengths.append(float(e["turn_length_um"]))
            all_turn_angles.append(float(e["turn_angle_deg"]))

        all_run_summaries.append({
            "run":                  run_index,
            "num_turns":            num_turns,
            "distance_traveled_um": float(dist_um[-1]),
            "turn_frequency":       float(turn_freq),
            "avg_speed_um_s":       float(avg_speed),
        })

        if hasattr(progress_bar, 'set_postfix'):   # update progress bar display with current run stats
            progress_bar.set_postfix(turns=num_turns, dist=f"{dist_um[-1]:.1f}um",
                                     tf=f"{turn_freq:.3f}")

    # --- Write batch-level summary files after all runs complete ---

    # One row per run, all runs combined
    with open(turn_freq_path, "w", newline="") as f:
        writer = csv.DictWriter(f, ["run", "num_turns", "distance_traveled_um",
                                     "turn_frequency", "avg_speed_um_s"])
        writer.writeheader()
        writer.writerows(all_run_summaries)

    lengths_arr = np.asarray(all_turn_lengths)   # convert pooled lists to arrays for statistics
    angles_arr  = np.asarray(all_turn_angles)

    # Compute aggregate statistics across all runs
    aggregate = {
        "total_turns":        int(lengths_arr.size),   # total number of binding events across all runs
        "avg_turn_length_um": float(np.mean(lengths_arr)) if lengths_arr.size else float("nan"),
        "std_turn_length_um": float(np.std(lengths_arr))  if lengths_arr.size else float("nan"),
        "avg_turn_angle_deg": float(np.mean(angles_arr))  if angles_arr.size  else float("nan"),
        "std_turn_angle_deg": float(np.std(angles_arr))   if angles_arr.size  else float("nan"),
        "avg_turn_frequency": float(np.mean(all_turn_frequencies)) if all_turn_frequencies else float("nan"),
        "avg_speed_um_s":     float(np.mean(all_average_speeds))   if all_average_speeds   else float("nan"),
    }

    with open(all_stats_path, "w", newline="") as f:
        writer = csv.DictWriter(f, list(aggregate.keys()))
        writer.writeheader()
        writer.writerow(aggregate)   # single row of aggregate stats

    plot_all_turn_lengths_hist(lengths_arr, all_hist_path, 60)   # log-log pooled histogram

    # Combined trackpy-format CSV for all runs
    with open(frame_csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, ["frame", "mass", "x", "y", "Majoraxis",
                                     "Minoraxis", "frame.1", "particle"])
        writer.writeheader()
        writer.writerows(all_frame_rows)

    # Fit and save power-law exponent from pooled bound durations
    durations_arr = np.asarray(all_bound_durations, float)
    alpha_mle, n_fit = fit_powerlaw_exponent(durations_arr, t_min=TIME_STEP * 2)
    with open(powerlaw_path, "w", newline="") as f:
        writer = csv.DictWriter(f, ["alpha_mle", "n_samples", "t_min_s",
                                     "total_bound_events", "note"])
        writer.writeheader()
        writer.writerow({
            "alpha_mle":           float(alpha_mle),
            "n_samples":           int(n_fit),
            "t_min_s":             float(TIME_STEP * 2),
            "total_bound_events":  int(durations_arr.size),
            "note":                "Clauset MLE; target range 1<alpha<3 for Levy-like walks",
        })
    print(f"Power-law exponent (MLE): alpha = {alpha_mle:.3f}  (n={n_fit})")
    print(f"Target range: 1.0 < alpha < 3.0 for Levy-like walks")

    # Final animation of SprBs on tracks (one video for the whole batch)
    animate_sprbs_on_tracks_gif(
        os.path.join(output_base_dir, "sprbs_on_tracks.mp4"),
        seed=0, gif_duration_s=10.0, fps=20, max_frames=250,
    )

    print(f"Saved outputs to {output_base_dir}/")

[DRAG] GAMMA_ROT_ROLL = 4.7785e-21 N·m·s
[DRAG] GAMMA_ROT      = 1.1396e-19 N·m·s
[DRAG] Ratio GAMMA_ROT / GAMMA_ROT_ROLL = 23.85x


In [173]:
# run_batch(n_runs=5, output_base_dir="simulation_output_mar15", make_movies=True, plot_track_once=True)

In [174]:
import os
import csv
from datetime import datetime

def compile_frame_csvs(output_base_dir):
    """
    Walks all run_XXX subdirectories in output_base_dir,
    collects every frame_csv.csv, tags each row with run index
    and a timestamp, and writes a single combined CSV.
    """
    compiled_rows = []
    fields = ["run", "timestamp", "frame", "mass", "x", "y",
              "Majoraxis", "Minoraxis", "frame.1", "particle"]

    run_dirs = sorted([
        d for d in os.listdir(output_base_dir)
        if d.startswith("run_") and
        os.path.isdir(os.path.join(output_base_dir, d))
    ])

    for run_dir in run_dirs:
        run_index = int(run_dir.split("_")[1])
        fcsv = os.path.join(output_base_dir, run_dir, "frame_csv.csv")
        if not os.path.exists(fcsv):
            print(f"  Warning: no frame_csv.csv in {run_dir}, skipping")
            continue

        with open(fcsv, newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                compiled_rows.append({
                    "run":       run_index,
                    "timestamp": f"run{run_index:03d}_frame{int(row['frame']):05d}",
                    "frame":     row["frame"],
                    "mass":      row["mass"],
                    "x":         row["x"],
                    "y":         row["y"],
                    "Majoraxis": row["Majoraxis"],
                    "Minoraxis": row["Minoraxis"],
                    "frame.1":   row["frame.1"],
                    "particle":  row["particle"],
                })

    out_path = os.path.join(output_base_dir, "all_frame_csv_combined.csv")
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(compiled_rows)

    print(f"Combined {len(compiled_rows)} rows from {len(run_dirs)} runs")
    print(f"Saved to: {out_path}")
    return out_path

# --- run ---
run_batch(
    n_runs=5,
    output_base_dir="simulation_output_apr28",
    make_movies=True,
    plot_track_once=True
)

compile_frame_csvs("simulation_output_apr28")

Simulation runs: 100%|█| 5/5 [00:38<00:00,  7.69s/it, dist=146.3um, tf=130.804, 


Power-law exponent (MLE): alpha = 10.354  (n=1341)
Target range: 1.0 < alpha < 3.0 for Levy-like walks
Saved outputs to simulation_output_apr28/
Combined 8335 rows from 5 runs
Saved to: simulation_output_apr28/all_frame_csv_combined.csv


'simulation_output_apr28/all_frame_csv_combined.csv'